# Olist E-commerce Analysis — Data Quality Audit

## Objective

The purpose of this notebook is to evaluate the quality of the raw Olist datasets before cleaning and analysis. The audit covers data types, missing values, duplicate records, key validity, referential integrity, timestamp consistency, and unusual numeric values.

In [1]:
import pandas as pd
from pathlib import Path


Raw_Data_Path = Path("../data/raw")

print(Raw_Data_Path.resolve())

C:\Users\Hp\DataAnalyst\data\raw


### 1. Orders Table Audit

#### 1.1 Dataset shape and structure
#### 1.2 Missing values
#### 1.3 Primary-key validation
#### 1.4 Order-status validation
#### 1.5 Timestamp consistency

In [2]:
orders = pd.read_csv(Raw_Data_Path/"olist_orders_dataset.csv")
orders.head(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [3]:
orders['order_status'].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

#### 1.1 Dataset shape and structure

In [4]:
orders.shape
print(f"Number of rows:{orders.shape[0]}")
print(f"Number of columns:{orders.shape[1]}")

Number of rows:99441
Number of columns:8


In [5]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [6]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [7]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


### Interpretation

The `orders` table contains **99,441 rows and 8 columns**. Each row is expected to represent one high-level order; however, the number of unique orders cannot be confirmed until the uniqueness of `order_id` is validated.

The `order_id` column is the candidate primary key of the `orders` table. It connects to `order_items.order_id`, where it acts as a foreign key. This is expected to form a one-to-many relationship because one order can contain multiple items.

The `customer_id` column in the `orders` table is a foreign key referencing the primary key `customer_id` in the `customers` table.

The `order_id`, `customer_id`, and `order_status` columns are appropriately stored as strings. However, the five order timestamp columns are also stored as strings and should eventually be converted to a datetime data type.

The initial structural review indicates missing values in `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date`. These values should not be immediately filled or removed. Their relationship with `order_status` must first be investigated to determine whether the missing timestamps are expected or represent data-quality issues.


#### 1.2 Missing Values

In [8]:
mv=orders.isna().sum().sum()
mv

np.int64(4908)

In [9]:
orders_missing_count = orders.isna().sum()

orders_missing_percentage = round((orders_missing_count / len(orders)) * 100, 2)

orders_missing_summary = pd.DataFrame({"missing_count": orders_missing_count,"missing_percentage": orders_missing_percentage})

orders_missing_summary = orders_missing_summary[orders_missing_summary["missing_count"] > 0].sort_values(by="missing_percentage",ascending=False)

orders_missing_summary

,missing_count,missing_percentage
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16


#### Missing-Value Findings

The missing-value audit identified null values in three timestamp columns:

* `order_delivered_customer_date` contains **2,965 missing values (2.98%)**.
* `order_delivered_carrier_date` contains **1,783 missing values (1.79%)**.
* `order_approved_at` contains **160 missing values (0.16%)**.

The remaining columns do not contain missing values. However, these null timestamps should not yet be classified as data-quality errors because their validity may depend on the corresponding `order_status`. Further investigation is required before deciding whether the records should be retained, excluded from specific calculations, or otherwise treated during cleaning.


#### 1.2.1 Missing Customer-Delivery Date Investigation

Missing timestamps do not automatically indicate data-quality errors because some order statuses may not have reached the approval, carrier-delivery, or customer-delivery stages. Therefore, the missing timestamp records will be grouped by `order_status` to determine whether the missing values are expected or require further investigation.


In [10]:
missing_customer_delivery_status = orders.loc[orders['order_delivered_customer_date'].isna(), ['order_status'] ]

missing_customer_delivery_status.value_counts()

order_status
shipped         1107
canceled         619
unavailable      609
invoiced         314
processing       301
delivered          8
created            5
approved           2
Name: count, dtype: int64

In [11]:
delivered_missing_customer_date = orders.loc[(orders['order_status']=="delivered") & (orders['order_delivered_customer_date'].isna())]

delivered_missing_customer_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


#### Interpretation and Treatment Decision

The `order_delivered_customer_date` column contains **2,965 missing values (2.98% of all orders)**. Most of these missing values are consistent with the corresponding order lifecycle stages:

* `shipped` orders have left the seller but do not yet have confirmed customer delivery.
* `created`, `approved`, `invoiced`, and `processing` orders have not reached the delivery stage.
* `canceled` and `unavailable` orders may never reach the customer.

However, **8 orders are marked as `delivered` while their customer-delivery timestamp is missing**. These records are classified as data-quality exceptions because a delivered order should normally have an actual delivery date. They represent approximately **0.0083% of all delivered orders**, indicating an isolated rather than systemic issue.

These eight orders will remain eligible for completed-sales analysis because sales metrics depend on the delivered status, order items, and prices. However, they will be flagged and excluded from delivery-duration, delay, and on-time-delivery calculations because those metrics require an actual customer-delivery timestamp. The number of excluded records will be documented when reporting delivery-related metrics.


##### 1.2.2 Missing Carrier-Delivery Date Investigation
Orders with a missing `order_delivered_carrier_date` will be grouped by `order_status` to determine whether the missing values are consistent with the order lifecycle or represent data-quality exceptions.


In [12]:
missing_carrier_date_status = orders.loc[orders['order_delivered_carrier_date'].isna(), ['order_status']]

missing_carrier_date_status.value_counts()

order_status
unavailable     609
canceled        550
invoiced        314
processing      301
created           5
approved          2
delivered         2
Name: count, dtype: int64

In [13]:
delivered_missing_carrier_date = orders.loc[(orders['order_status']=="delivered") & (orders['order_delivered_carrier_date'].isna())]

delivered_missing_carrier_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


##### Carrier-Delivery Date Findings

The `order_delivered_carrier_date` column contains **1,783 missing values**. Most occur in `unavailable`, `canceled`, `invoiced`, `processing`, `created`, and `approved` orders, which may not have reached the carrier-delivery stage.

However, 2 orders are marked as delivered while their carrier-delivery timestamp is missing**. These records are classified as isolated data-quality exceptions. They can remain in completed-sales analysis but should be flagged and excluded from calculations requiring the carrier-delivery timestamp, such as carrier-to-customer transit time.


#### 1.2.3 Missing Approval Date Investigation

In [14]:
missing_approval_status = orders.loc[ orders['order_approved_at'].isna(), ['order_status']]

missing_approval_status.value_counts()

order_status
canceled        141
delivered        14
created           5
Name: count, dtype: int64

In [15]:
delivered_missing_approval_date = orders.loc[ (orders['order_status']=="delivered") & (orders['order_approved_at'].isna()) ]

delivered_missing_approval_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00


##### Approval Date Findings

The `order_approved_at` column contains **160 missing values**: 141 belong to `canceled` orders, 14 to `delivered` orders, and 5 to `created` orders. Missing approval timestamps for `created` orders and some `canceled` orders may be consistent with orders that did not complete the approval stage.

However, **14 orders are marked as delivered while their approval timestamp is missing**. These records are classified as isolated data-quality exceptions because a completed delivered order would normally have an approval timestamp.

The 14 orders can remain in completed-sales analysis because the approval timestamp is not required to calculate sales. However, they should be flagged and excluded from calculations that require `order_approved_at`, such as purchase-to-approval time or approval-to-carrier time.


### 1.3 Primary-Key Validation

The `order_id` column is expected to uniquely identify each row in the `orders` table. To determine whether it can serve as the primary key, it will be checked for missing values, duplicate values, and consistency between the total number of rows and the number of distinct order IDs.


In [16]:
missing_order_id = orders.loc[ orders['order_id'].isna()]
missing_order_id

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [17]:
total_order_id = len(orders)
total_distinct_order_id = orders['order_id'].nunique()

if total_order_id == total_distinct_order_id:
    print(f"Number of orders: {total_order_id}")
    print(f"Number of distinct orders: {total_distinct_order_id}")
    print("Both are equal")
else:
    print(f"Number of orders: {total_order_id}")
    print(f"Number of distinct orders: {total_distinct_order_id}")
    print(f"Duplicate order_id's may be present")


Number of orders: 99441
Number of distinct orders: 99441
Both are equal


In [18]:
duplicate_values = orders['order_id'].duplicated()

duplicate_values.value_counts()

order_id
False    99441
Name: count, dtype: int64

#### Primary-Key Findings

The `orders` table contains **99,441 rows and 99,441 distinct `order_id` values**. The `order_id` column contains no missing values, and the duplicate check returned **99,441 `False` values and no `True` values**, confirming that the duplicate count is zero.

Therefore, every row has a unique and non-null `order_id`, confirming that `order_id` is a valid primary key for the `orders` table.


### 1.4 Order-Status Validation

The `order_status` column represents the current stage of each order in the order lifecycle. It will be validated by examining the distinct status values, their frequency distribution, and the presence of any missing, blank, inconsistent, or unexpected categories.


In [19]:
distinct_order_status_count = orders['order_status'].nunique()

distinct_order_status_count

8

In [20]:
each_order_status_count =  orders['order_status']

each_order_status_count.value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [21]:
missing_order_status_count = orders["order_status"].isna().sum()
missing_order_status_count

np.int64(0)

In [22]:
blank_order_status_count = orders["order_status"].str.strip().eq("").sum()
blank_order_status_count

np.int64(0)

#### Order-Status Findings

The `order_status` column contains **8 distinct categories**, and their frequency counts account for all **99,441 orders**. The categories are `delivered`, `shipped`, `canceled`, `unavailable`, `invoiced`, `processing`, `created`, and `approved`.

The validation found **no missing values and no empty or whitespace-only values**. The status categories are consistently formatted, and no unexpected categories were identified. Therefore, no cleaning treatment is required for `order_status`.

The `delivered` category contains 96,478 orders and will be used as the primary completed-sales filter in the subsequent analysis.


### 1.5 Timestamp Consistency

The order lifecycle is represented by five timestamp columns: `order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, and `order_estimated_delivery_date`.

These columns will be converted to datetime in a separate audit copy of the Orders table. We will verify whether the conversion introduces additional missing values and check whether the timestamps follow the expected chronological order of the order lifecycle.


#### 1.5.1 Datetime Conversion Validation

The five order timestamp columns will be converted from string to datetime in a separate audit copy. Missing-value counts before and after conversion will be compared to determine whether any invalid timestamp values are converted into additional `NaT` values.


In [23]:
orders_timestamp_audit = pd.read_csv(Raw_Data_Path/"olist_orders_dataset.csv")
orders_timestamp_audit.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [24]:
#columns need to be coverted
datetime_columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

#converting to datetimeformat
for col in datetime_columns:
    orders_timestamp_audit[col] = pd.to_datetime( orders_timestamp_audit[col], errors='coerce')

orders_timestamp_audit.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [25]:
datetime_columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

#missing values data after coversion 
after_conversion = orders_timestamp_audit[datetime_columns].isna().sum()
after_conversion

#missing values data before coversion
before_conversion = orders[datetime_columns].isna().sum()
after_conversion

missing_datetime_comparison_summary = pd.DataFrame({"before_conversion":before_conversion,"after_conversion":after_conversion})

missing_datetime_comparison_summary.sort_values(by="after_conversion", ascending=False )

,before_conversion,after_conversion
order_delivered_customer_date,2965,2965
order_delivered_carrier_date,1783,1783
order_approved_at,160,160
order_purchase_timestamp,0,0
order_estimated_delivery_date,0,0


**Finding:** The missing-value counts are identical before and after datetime conversion. Therefore, the conversion produced no additional `NaT` values, indicating that all non-missing timestamp values were successfully converted to datetime.


#### 1.5.2 Order Lifecycle Sequence Validation

The order timestamps will be checked against the expected lifecycle sequence to identify records containing logically inconsistent dates, such as approval occurring before purchase or customer delivery occurring before carrier delivery.


In [26]:
approval_before_purchase = orders_timestamp_audit.loc[ orders_timestamp_audit['order_approved_at'] < orders_timestamp_audit['order_purchase_timestamp']]

approval_before_purchase.shape

(0, 8)

In [27]:
carrier_before_purchase = orders_timestamp_audit.loc[ orders_timestamp_audit['order_delivered_carrier_date'] < orders_timestamp_audit['order_purchase_timestamp']]

carrier_before_purchase.shape

(166, 8)

In [28]:
order_status_carrier_before_purchase = carrier_before_purchase.loc[ carrier_before_purchase['order_status']=="delivered"]
order_status_carrier_before_purchase.shape

(165, 8)

In [29]:
carrier_before_purchase['order_status'].value_counts()

order_status
delivered    165
shipped        1
Name: count, dtype: int64

In [30]:
carrier_before_purchase['carrier_purchase_time_gap'] = carrier_before_purchase['order_purchase_timestamp'] - carrier_before_purchase['order_delivered_carrier_date']
carrier_before_purchase.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,carrier_purchase_time_gap
615,b9afddbdcfadc9a87b41a83271c3e888,85c6af75161b8b2b1af98e82b5a6a5a5,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04,0 days 00:23:48
1111,ad133696906f6a78826daa0911b7daec,e6f5b234bb0d847f10eebd70130c5d49,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18,0 days 00:49:22
1329,74e033208dc13a7b8127eb8e73d09b76,72fcbb1145f2889eddcba6d6c1d6c090,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29,0 days 00:59:44
1372,a6b58794fd2ba533359a76c08df576e3,ccd8e3459ad58ae538c9d2cf35532ba4,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08,0 days 01:32:23
1864,5792e0b1c8c8a2bf53af468c9a422c88,5c427f06e7fcf902e309743516e2c580,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09,0 days 00:43:14


In [31]:
carrier_before_purchase.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,carrier_purchase_time_gap
count,166,166,166,165,166,166
mean,2018-07-01 11:02:58.987951,2018-07-01 12:56:11.590361,2018-06-30 09:00:36.867470,2018-07-09 04:20:30.236363,2018-07-23 14:18:47.710843,1 days 02:02:22.120481
min,2018-04-25 13:02:26,2018-04-25 13:15:26,2018-01-26 13:35:00,2018-04-27 17:24:52,2018-05-09 00:00:00,0 days 00:00:24
25%,2018-06-06 20:29:19.500000,2018-06-06 20:44:51.250000,2018-06-05 19:04:00,2018-06-14 22:18:58,2018-07-05 00:00:00,0 days 00:14:04
50%,2018-06-30 00:50:13.500000,2018-06-30 01:10:56,2018-06-29 15:17:00,2018-07-09 15:52:25,2018-07-25 00:00:00,0 days 00:35:59.500000
75%,2018-08-03 14:32:12.250000,2018-08-03 14:48:38.250000,2018-08-03 13:56:30,2018-08-09 16:40:37,2018-08-17 00:00:00,0 days 01:01:03
max,2018-08-24 14:37:50,2018-08-24 14:50:14,2018-08-24 13:46:00,2018-08-30 20:47:26,2018-10-05 00:00:00,171 days 05:05:53
std,NaN,NaN,NaN,NaN,NaN,13 days 06:55:25.842359


In [32]:
large_carrier_purchase_gaps = carrier_before_purchase.loc[carrier_before_purchase['carrier_purchase_time_gap'].dt.days >= 1]

large_carrier_purchase_gaps

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,carrier_purchase_time_gap
25883,7c48bb55e8e4f7e56d412e9653db37bc,34ef6181341eb36c47fd601c46878f00,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07,171 days 05:05:53
83321,4021cd7611d6d9ce5ffcd24817fc374f,ac27243bffbc9c58eb62adfb58c71488,delivered,2018-08-18 11:49:40,2018-08-18 12:10:30,2018-08-14 06:22:00,2018-08-29 04:41:53,2018-08-30,4 days 05:27:40


In [33]:
customer_before_carrier = orders_timestamp_audit.loc[ orders_timestamp_audit['order_delivered_customer_date'] < orders_timestamp_audit['order_delivered_carrier_date']]

len(customer_before_carrier)

23

In [34]:
customer_before_carrier_order_status = customer_before_carrier.loc[ customer_before_carrier['order_status'] == "delivered"]
len(customer_before_carrier_order_status)

23

In [35]:
customer_before_carrier['customer_carrier_time_gap'] = customer_before_carrier['order_delivered_carrier_date'] - customer_before_carrier['order_delivered_customer_date']
customer_before_carrier.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_carrier_time_gap
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14,2 days 21:25:02
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21,1 days 02:54:50
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08,1 days 05:08:34
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11,6 days 00:14:20
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31,0 days 04:11:01


In [36]:
customer_before_carrier.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_carrier_time_gap
count,23,23,23,23,23,23
mean,2017-05-01 18:35:07.086956,2017-05-02 05:49:13.956521,2017-05-11 21:18:35.521739,2017-05-08 14:51:17.043478,2017-06-01 09:23:28.695652,3 days 06:27:18.478260
min,2016-10-07 10:05:16,2016-10-07 11:24:43,2016-10-21 18:02:40,2016-10-20 18:03:17,2016-11-29 00:00:00,0 days 00:23:18
25%,2017-02-07 14:40:21.500000,2017-02-07 15:00:42.500000,2017-02-25 10:03:17,2017-02-14 21:34:51.500000,2017-03-29 00:00:00,0 days 23:20:18
50%,2017-07-07 11:22:13,2017-07-07 12:25:07,2017-07-11 18:15:40,2017-07-11 17:41:26,2017-07-21 00:00:00,1 days 15:51:52
75%,2017-07-20 16:09:28.500000,2017-07-21 21:14:12.500000,2017-07-30 17:37:52.500000,2017-07-26 06:56:10,2017-08-14 12:00:00,5 days 08:45:32.500000
max,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-12 00:03:35,2017-08-10 21:22:31,2017-09-05 00:00:00,16 days 02:18:29
std,NaN,NaN,NaN,NaN,NaN,3 days 17:18:29.050924


In [37]:
estimated_before_purchase = orders_timestamp_audit.loc[ orders_timestamp_audit['order_estimated_delivery_date'] < orders_timestamp_audit['order_purchase_timestamp']]
len(estimated_before_purchase)

0

In [38]:
carrier_before_approval = orders_timestamp_audit.loc[ orders_timestamp_audit['order_delivered_carrier_date'] < orders_timestamp_audit['order_approved_at']]
len(carrier_before_approval)

1359

In [39]:
carrier_before_approval_order_status = carrier_before_approval.loc[ carrier_before_approval['order_status'] == "delivered"]
len(carrier_before_approval_order_status)

1350

In [40]:
carrier_before_approval['order_status'].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

In [41]:
carrier_before_approval['carrier_approval_time_gap'] = carrier_before_approval['order_approved_at'] - carrier_before_approval['order_delivered_carrier_date']
carrier_before_approval

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,carrier_approval_time_gap
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04,1 days 08:37:02
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15,0 days 23:06:08
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31,2 days 10:34:53
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31,0 days 00:07:53
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06,3 days 09:28:09
...,...,...,...,...,...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,fcdd7556401aaa1c980f8b67a69f95dc,delivered,2018-07-02 16:30:02,2018-07-05 16:17:59,2018-07-05 14:11:00,2018-07-10 23:21:47,2018-07-24,0 days 02:06:59
99230,78008d03bd8ef7fcf1568728b316553c,043e3254e68daf7256bda1c9c03c2286,delivered,2018-07-03 13:11:13,2018-07-05 16:32:52,2018-07-03 12:57:00,2018-07-10 17:47:39,2018-07-23,2 days 03:35:52
99266,76a948cd55bf22799753720d4545dd2d,3f20a07b28aa252d0502fe7f7eb030a9,delivered,2018-01-30 02:41:30,2018-02-04 23:31:46,2018-01-31 18:11:58,2018-03-18 20:08:50,2018-03-02,4 days 05:19:48
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,6d63fa86bd2f62908ad328325799152f,delivered,2018-04-20 17:28:40,2018-04-24 19:26:10,2018-04-23 17:18:40,2018-04-28 17:38:42,2018-05-15,1 days 02:07:30


In [42]:
carrier_before_approval.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,carrier_approval_time_gap
count,1359,1359,1359,1350,1359,1359
mean,2018-05-17 17:50:08.729948,2018-05-20 00:21:19.643855,2018-05-18 23:36:12.490802,2018-05-26 09:58:51.852592,2018-06-09 16:11:39.337748,1 days 00:45:07.153053
min,2017-02-19 01:15:03,2017-03-01 10:51:46,2017-02-22 16:05:29,2017-02-24 14:27:26,2017-03-17 00:00:00,0 days 00:00:21
25%,2018-04-22 11:55:27.500000,2018-04-24 18:53:37.500000,2018-04-23 23:12:08.500000,2018-04-30 22:06:16.250000,2018-05-17 00:00:00,0 days 01:24:55.500000
50%,2018-06-29 14:24:39,2018-07-05 16:01:00,2018-06-30 08:58:00,2018-07-05 14:14:21,2018-07-17 00:00:00,0 days 17:10:04
75%,2018-07-04 15:19:43.500000,2018-07-05 16:32:45,2018-07-05 14:04:00,2018-07-11 20:37:59.250000,2018-07-31 00:00:00,1 days 01:57:24
max,2018-08-25 09:46:04,2018-08-27 13:15:03,2018-08-27 10:43:00,2018-09-19 15:46:39,2018-10-05 00:00:00,171 days 05:15:22
std,NaN,NaN,NaN,NaN,NaN,4 days 19:18:28.055754


In [43]:
long_gap_carrier_before_approval= carrier_before_approval.loc[carrier_before_approval['carrier_approval_time_gap'].dt.days >= 1]
long_gap_carrier_before_approval.sort_values(by='carrier_approval_time_gap', ascending=False)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,carrier_approval_time_gap
25883,7c48bb55e8e4f7e56d412e9653db37bc,34ef6181341eb36c47fd601c46878f00,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07,171 days 05:15:22
14562,1fab4ac9d85079b3da72a11475ae1685,f831c1fa80308975ec2b58e4877328e0,delivered,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-04 13:10:23,2017-09-08 20:13:03,2017-09-20,9 days 08:55:48
46163,0184d4ddb259e1a4cfc2871888cf97b8,09425ea1839abf2f0d289a0ff453fa21,delivered,2017-09-01 20:04:28,2017-09-13 22:17:15,2017-09-04 14:05:50,2017-09-09 15:12:44,2017-09-20,9 days 08:11:25
98710,1378f9601350615613cc8832d6789c5d,988126b4ddf725d9724e4318872ea2ae,delivered,2017-09-01 20:28:02,2017-09-13 22:03:51,2017-09-04 18:07:55,2017-09-13 22:24:46,2017-09-29,9 days 03:55:56
41592,8554cb37f7158cb0b082a841d24a4589,788e845925ff64c9df5d8ba40e28cf50,delivered,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,2017-09-08 20:07:45,2017-10-02,9 days 02:45:45
...,...,...,...,...,...,...,...,...,...
61593,4e790ef1ee5dca2f0de38704d806e45d,2bc33799adeccec1a65fa254bdec653d,delivered,2018-06-30 01:23:27,2018-07-05 16:05:22,2018-07-04 16:03:00,2018-07-11 20:40:27,2018-08-08,1 days 00:02:22
87866,e55507dc5f963fbdb8ae1c47181974b5,eb9f81343ccb0361b66606660a0941c2,delivered,2018-04-22 12:36:57,2018-04-24 18:48:15,2018-04-23 18:46:53,2018-05-05 15:38:46,2018-05-25,1 days 00:01:22
33125,d04e58d009641c790ddf6b7b41790f87,59a7da17a354b4ef42f557e5fb5112af,delivered,2018-04-22 21:00:26,2018-04-24 19:27:39,2018-04-23 19:26:39,2018-04-25 20:08:39,2018-05-14,1 days 00:01:00
59523,22a0f64560b6a90eb7175a87387a1aa1,5e5d0d5a01a3e4809c36751fd5f0a6ff,delivered,2018-04-21 14:36:39,2018-04-24 18:04:42,2018-04-23 18:03:44,2018-05-03 19:36:32,2018-05-24,1 days 00:00:58


#### 1.5.3 Timestamp Consistency Findings and Treatment Decision

All five timestamp columns were successfully converted from string to datetime. The missing-value counts remained unchanged after conversion, confirming that no additional `NaT` values were introduced.

The order-lifecycle sequence validation produced the following findings:

* **Approval before purchase:** No records were found.
* **Carrier delivery before approval:** 1,359 records (approximately 1.37% of all orders) were identified. Of these, 1,350 were delivered and 9 were shipped. The median gap was approximately 17 hours, while the maximum gap was approximately 171 days.
* **Carrier delivery before purchase:** 166 records (approximately 0.17% of all orders) were identified. Of these, 165 were delivered and 1 was shipped. Approximately 75% of the gaps were one hour or less, but two large-gap exceptions of approximately 4 days and 171 days were found.
* **Customer delivery before carrier delivery:** 23 records (approximately 0.02% of all orders) were identified, and all were marked as delivered. The median gap was approximately 1 day and 16 hours, while the maximum gap was approximately 16 days.
* **Estimated delivery before purchase:** No records were found.

These exception counts should not be added together because some records appear in more than one condition. In particular, the 166 carrier-before-purchase records are included within the carrier-before-approval exceptions.

The inconsistent timestamps will not be overwritten or imputed because the correct event times cannot be reliably inferred from the available data. These orders may remain in completed-sales analysis when their order, item, and price information is otherwise valid. However, they should be flagged and excluded from lifecycle-duration calculations that depend on the affected timestamp sequence, including purchase-to-approval, approval-to-carrier, and carrier-to-customer duration metrics.


## 2. Customers Table Audit
The Customers table connects each order-specific customer record with a persistent customer identifier and geographic information. This audit evaluates the table’s structure, completeness, identifier uniqueness, customer-identity relationships, and location-field consistency before the data is used for customer and regional analysis.

### 2.1 Dataset Shape and Structure

This subsection examines the number of records and columns, column meanings, and current data types. It also identifies fields that require additional validation before they can be treated as keys or used in analysis.


In [44]:
customers = pd.read_csv(Raw_Data_Path/"olist_customers_dataset.csv")
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [45]:
customers.shape

(99441, 5)

In [46]:
customers.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')

In [47]:
customers.dtypes

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

In [48]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


**Initial observation:** The Customers table contains 99,441 rows and 5 columns. Each row is expected to represent one order-specific customer record. `customer_id` is the candidate primary key, while `customer_unique_id` is intended to identify the same customer across multiple orders. The geographic fields contain the customer’s ZIP-code prefix, city, and state. Although `customer_zip_code_prefix` is stored as an integer, it represents a geographic identifier rather than a numeric measurement.


### 2.2 Missing-Value Validation

Missing-value counts and percentages will be calculated for every column to confirm whether the customer identifiers and geographic attributes are complete.


In [49]:
customers_missing_count = customers.isna().sum()
customers_missing_percentage = (customers_missing_count/ len(customers)) * 100
customers_missing_summary = pd.DataFrame({"missing_count":customers_missing_count, "missing_percentage":customers_missing_percentage})
customers_missing_summary

,missing_count,missing_percentage
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


**Finding:** No missing values were found in any of the five Customers-table columns. Both customer identifiers and all geographic attributes are complete, so no missing-value treatment is required.


### 2.3 Customer Identifier Validation

The uniqueness and completeness of `customer_id` will be validated to determine whether it can serve as the Customers table’s primary key. The relationship between `customer_id` and `customer_unique_id` will also be examined because one persistent customer may be associated with multiple order-specific customer IDs.


In [50]:
#total number of rows
print(f"total_number_of_rows: {customers.shape[0]}")


total_number_of_rows: 99441


In [51]:
#Number of distinct customer_id values
customers_distinct_customer_id = customers['customer_id'].unique()
print(f"Distinct customer_id count: {customers_distinct_customer_id.shape[0]}")

Distinct customer_id count: 99441


In [52]:
#Number of duplicated customer_id values
customers_duplicated_customer_id = customers['customer_id'].duplicated().sum()
print(f"Duplicated customer_id count: {customers_duplicated_customer_id}")


Duplicated customer_id count: 0


In [53]:
#Total distinct customer_unique_id values
customers_distinct_customer_unique_id = customers['customer_unique_id'].unique()
print(f"Distinct customer_unique_id count: {customers_distinct_customer_unique_id.shape[0]}")

Distinct customer_unique_id count: 96096


In [54]:
#Number of repeated occurrences
customers_duplicated_customer_unique_id = customers['customer_unique_id'].duplicated().sum()
print(f"Duplicated customer_unique_id count: {customers_duplicated_customer_unique_id}")


Duplicated customer_unique_id count: 3345


In [55]:
#Number of persistent customers appearing more than once
customer_unique_id_frequency = customers['customer_unique_id'].value_counts()
repeat_customer_unique_ids = customer_unique_id_frequency[customer_unique_id_frequency>1]
print(f"Number of entries in repeat_customer_unique_ids: {len(repeat_customer_unique_ids)}")
print(f"Max orders_ids associated with one customer_unique_id: {max(repeat_customer_unique_ids)}")


Number of entries in repeat_customer_unique_ids: 2997
Max orders_ids associated with one customer_unique_id: 17


#### Customer Identifier Findings

The Customers table contains 99,441 rows, 99,441 distinct `customer_id` values, no missing customer IDs, and no duplicated customer IDs. Therefore, `customer_id` is a valid primary key for the Customers table.

The table contains 96,096 distinct `customer_unique_id` values. A total of 2,997 persistent customers are associated with more than one order-specific `customer_id`, and one persistent customer is associated with a maximum of 17 customer records.

The repeated `customer_unique_id` values are expected and should not be removed as duplicates. `customer_id` should be used to join Customers with Orders, while `customer_unique_id` should be used for repeat-customer and customer-level analysis.


### 2.4 Geographic Consistency Validation

The customer ZIP-code prefix, city, and state fields will be examined for invalid formats, blank values, unexpected state codes, and inconsistent geographic mappings. Multiple locations associated with the same persistent customer will be treated as possible customer movement rather than automatically classified as duplicate or erroneous records.


In [56]:
#Number of distinct customer_state values
print(f"Number of distinct customer_state: {len(customers['customer_state'].unique())}")


Number of distinct customer_state: 27


In [57]:
#each state and its frequency
customers['customer_state'].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [58]:
#Checking blank or whitespace-only values for customer_state
customer_state_blank = customers[customers['customer_state'].str.strip() == '']
len(customer_state_blank)

0

In [59]:
#Does not contain exactly two characters, or Is not entirely uppercase 
invalid_customer_state_format = customers.loc[ (customers['customer_state'].str.len() != 2) | (~customers['customer_state'].str.isupper())]
len(invalid_customer_state_format)

0

In [60]:
#Number of distinct cities 
customers['customer_city'].nunique()

4119

In [61]:
#Checking blank or whitespace-only values for customer_city 
customer_city_blank = customers[customers['customer_city'].str.strip() == '']
len(customer_city_blank)

0

In [62]:
#detect city values containing leading or trailing spaces
city_with_leading_space = customers['customer_city'].str.startswith(' ')
city_with_trailing_space = customers['customer_city'].str.endswith(' ')

city_has_space = customers.loc[city_with_leading_space | city_with_trailing_space]
len(city_has_space)

0

In [63]:
#Number of distinct ZIP-code prefixes
customers_distinct_zip_code = customers['customer_zip_code_prefix'].nunique()
print(f"Number of distinct zip_codes: {customers_distinct_zip_code}")

Number of distinct zip_codes: 14994


In [64]:
#Minimum and maximum values
max_values = max(customers['customer_zip_code_prefix'])
min_values = min(customers['customer_zip_code_prefix'])
print(f"max of city_zip_code: {max_values}")
print(f"min of city_zip_code: {min_values}")

max of city_zip_code: 99990
min of city_zip_code: 1003


In [65]:
#Number Minimum and maximum values
max_values = len(str(max(customers['customer_zip_code_prefix'])))
min_values = len(str(min(customers['customer_zip_code_prefix'])))
print(f"max length of a city_zip_code: {max_values}")
print(f"min length of a city_zip_code: {min_values}")

max length of a city_zip_code: 5
min length of a city_zip_code: 4


In [66]:
#Number of values below 0 or above 99,999
lessthan0_or_greaterthan99999 = len(customers.loc[(customers['customer_zip_code_prefix'] < 0) | (customers['customer_zip_code_prefix'] > 99999)])
print(f"Number of values below 0 or above 99,999: {lessthan0_or_greaterthan99999}")

Number of values below 0 or above 99,999: 0


In [67]:
#Count rows with customer_zip_code_prefix < 10000
customers_zip_code_lessthan_10000 = len(customers.loc[customers['customer_zip_code_prefix']<10000])
customers_zip_code_lessthan_10000

23995

In [68]:
#ZIP-code prefixes associated with more than one customer_state
((customers.groupby('customer_zip_code_prefix')['customer_state'].nunique())>1).sum()

np.int64(0)

#### Geographic Validation Findings

The Customers table contains 27 valid two-letter state codes, representing all Brazilian federative units. No blank, whitespace-only, lowercase, or incorrectly formatted state values were found.

The table contains 4,119 distinct customer-city labels. No blank city values or city names containing leading or trailing spaces were identified.

There are 14,994 distinct ZIP-code prefixes, ranging from 1,003 to 99,990. No values fall outside the expected numeric range, and no ZIP-code prefix is associated with more than one customer state, indicating internally consistent ZIP-to-state mappings.

However, 23,995 customer records have ZIP-code prefixes below 10,000. These values are not data-quality errors; their leading zeros were removed because the field was stored as an integer. During cleaning, `customer_zip_code_prefix` should be converted to a five-character string and padded with leading zeros where required.


## 3. Order Items Table Audit

The Order Items table contains product-level records associated with customer orders. Each row is expected to represent one item within an order and includes the related product, seller, shipping deadline, price, and freight value.

This audit evaluates the table’s structure, completeness, composite-key uniqueness, identifier validity, timestamp format, and numeric-value consistency before it is used for sales, product, seller, and delivery analysis.

### 3.1 Dataset Shape and Structure

This subsection examines the table’s dimensions, columns, data types, and record grain to confirm how individual items are represented within each order.


In [69]:
order_items = pd.read_csv(Raw_Data_Path/"olist_order_items_dataset.csv")
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [70]:
order_items.shape

(112650, 7)

In [71]:
order_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')

In [72]:
order_items.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [73]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


**Initial observation:** The Order Items table contains 112,650 item-level records and 7 columns. The number of item records is greater than the number of orders because one order may contain multiple products.

The identifier columns are stored as strings, `order_item_id` is stored as an integer, and the monetary fields `price` and `freight_value` are stored as floating-point numbers. However, `shipping_limit_date` is stored as a string and should be converted to datetime during preparation.

The combination of `order_id` and `order_item_id` is expected to uniquely identify each order-item record, but this must be validated before treating it as a composite primary key.


### 3.2 Missing-Value Validation

Missing-value counts and percentages will be calculated for every Order Items column to confirm whether the identifiers, shipping deadline, price, and freight values are complete.


In [74]:
order_items_missing_count = order_items.isna().sum()

order_items_missing_percentage = (order_items_missing_count/len(order_items))*100

order_items_missing_summary = pd.DataFrame({"missing_count":order_items_missing_count, "missing_percentage":order_items_missing_percentage})

order_items_missing_summary


,missing_count,missing_percentage
order_id,0,0.0
order_item_id,0,0.0
product_id,0,0.0
seller_id,0,0.0
shipping_limit_date,0,0.0
price,0,0.0
freight_value,0,0.0


**Finding:** No missing values were identified in any Order Items column. All order, product, and seller identifiers are complete, and every record contains a shipping deadline, price, and freight value. Therefore, no missing-value treatment is required for this table.


### 3.3 Composite-Key Validation

`order_id` is expected to repeat because one order may contain multiple items, and `order_item_id` is expected to identify the item’s sequence within each order. The combination of `order_id` and `order_item_id` will be tested for missing values and duplicate combinations to determine whether it forms a valid composite primary key.


In [75]:
#number of distinct order_id–order_item_id combinations
ot_orderid_orderiteid_noduplicates = order_items[['order_id','order_item_id']].drop_duplicates()
ot_distinct_orderid_orderiteid_pair = len(ot_orderid_orderiteid_noduplicates)
ot_distinct_orderid_orderiteid_pair

112650

In [76]:
#count duplicated order_id–order_item_id combinations using both columns together
int(order_items[['order_id', 'order_item_id']].duplicated().sum())

0

#### Composite-Key Findings

The Order Items table contains 112,650 rows and 112,650 distinct `order_id`–`order_item_id` combinations. No duplicated combinations or missing key values were identified.

Therefore, the combination of `order_id` and `order_item_id` forms a valid composite primary key for the Order Items table. `order_id` may repeat because one order can contain multiple items, while `order_item_id` distinguishes the individual item records within each order.


### 3.4 Order-Item Sequence Validation

The `order_item_id` field is expected to begin at 1 and increase sequentially within each order. This check will identify non-positive item IDs and orders containing gaps or inconsistencies in their item-number sequence.


In [77]:
#Minimum and maximum order_item_id
min_order_item_id = min(order_items['order_item_id'])
max_order_item_id = max(order_items['order_item_id'])
print(f"min order_item_id: {min_order_item_id}")
print(f"max order_item_id: {max_order_item_id}")


min order_item_id: 1
max order_item_id: 21


In [78]:
#Number of records where order_item_id is less than or equal to zero
len(order_items.loc[order_items['order_item_id'] == 0])


0

In [79]:
#Whether each order’s maximum order_item_id equals its number of item records
item_count_per_order = order_items.groupby('order_id')['order_item_id'].count()

max_item_count_per_order = order_items.groupby('order_id')['order_item_id'].max()

sequence_mismatch = item_count_per_order != max_item_count_per_order
int(sequence_mismatch.sum())

0

#### Order-Item Sequence Findings

`order_item_id` values range from 1 to 21, and no zero or negative values are present. For every order, the maximum `order_item_id` equals the number of item records associated with that order.

Therefore, item numbering begins at 1 and follows a complete sequential pattern without gaps or inconsistencies.


### 3.5 Numeric-Value Validation

The `price` and `freight_value` columns will be examined for negative values, zero values, and unusual numeric ranges. Zero freight values may represent free-shipping transactions and will not automatically be treated as errors, while negative monetary values would represent invalid records.


In [80]:
#Generate descriptive statistics for price and freight_value
order_items[['price', 'freight_value']].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [81]:
#Negative price values
len(order_items.loc[order_items['price'] < 0])

0

In [82]:
#Zero price values
len(order_items.loc[order_items['price'] == 0])

0

In [83]:
#Negative freight_value values
len(order_items.loc[order_items['freight_value'] < 0])

0

In [84]:
#Zero freight_value values
len(order_items.loc[order_items['freight_value'] == 0])

383

In [85]:
#Top five records with the highest price
order_items.sort_values(by='price', ascending=False).head()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,2017-02-16 20:37:36,6735.0,194.31
112233,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,2018-08-02 04:05:13,6729.0,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,2017-06-15 02:45:17,6499.0,227.66
74336,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,2017-04-18 13:25:18,4799.0,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,2017-05-09 15:50:15,4690.0,74.34


In [86]:
#Top five records with the highest freight_value
order_items.sort_values(by='freight_value', ascending=False).head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
73486,a77e1550db865202c56b19ddc6dc4d53,1,ec31d2a17b299511e7c8627be9337b9b,257e61d3251fb5efb9daadddbc2cf7ca,2018-04-25 06:55:13,979.00,409.68
28044,3fde74c28a3d5d618c00f26d51baafa0,1,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2018-08-22 03:15:09,2338.08,375.28
3303,076d1555fb53a89b0ef4d529e527a0f6,1,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2018-08-07 04:35:22,2338.08,375.28
69797,9f49bd16053df810384e793386312674,1,256a9c364b75753b97bee410c9491ad8,5c030029b5916fed0986310385ec9009,2018-08-02 03:44:28,1149.00,339.59
16731,264a7e199467906c0727394df82d1a6a,1,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,2018-04-23 17:15:10,1050.00,338.30


#### Numeric-Value Findings

Item prices range from 0.85 to 6,735.00, with a median of 74.99. No zero or negative prices were found. The highest-priced records contain valid identifiers and positive freight charges, so they will be retained as plausible high-value transactions.

Freight values range from 0.00 to 409.68, with a median of 16.26. No negative freight values were identified. A total of 383 records contain zero freight values; these are considered plausible free-shipping transactions and will be retained.

The highest freight charges are generally associated with relatively expensive products and do not show obvious formatting or sign errors. High price and freight values will not be removed solely because they are statistically unusual.


### 3.6 Shipping-Limit Date Validation

`shipping_limit_date` is currently stored as a string. It will be converted to datetime in a separate audit copy, and missing-value counts will be compared before and after conversion to identify invalid timestamp formats. The minimum and maximum dates will also be reviewed for unexpected ranges.


In [87]:
order_items_timestamp_audit = pd.read_csv(Raw_Data_Path/"olist_order_items_dataset.csv")
order_items_timestamp_audit.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [88]:
#converting shipping_limit_date to datetime format
order_items_timestamp_audit['shipping_limit_date'] = pd.to_datetime(order_items_timestamp_audit['shipping_limit_date'], errors='coerce')
order_items_timestamp_audit.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [89]:
#Comparing the missing count before and after conversion

order_items_before_conversion = order_items['shipping_limit_date'].isna().sum()
order_items_before_conversion

order_items_after_conversion = order_items_timestamp_audit['shipping_limit_date'].isna().sum()
order_items_after_conversion

order_items_shippingdate_missing_summary = pd.DataFrame({"before_conversion":order_items_before_conversion, "after_conversion":order_items_after_conversion}, index=['shipping_limit_date'])
order_items_shippingdate_missing_summary.sort_values(by="before_conversion", ascending=False)

,before_conversion,after_conversion
shipping_limit_date,0,0


In [90]:
#minimum and maximum values of the converted shipping_limit_date
print(f"Min shipping_limit_date: {min(order_items_timestamp_audit['shipping_limit_date'])}")
print(f"Max shipping_limit_date: {max(order_items_timestamp_audit['shipping_limit_date'])}")

Min shipping_limit_date: 2016-09-19 00:15:34
Max shipping_limit_date: 2020-04-09 22:35:08


In [91]:
#create shipping_limit_after_2018
shipping_limit_after_2018 = order_items_timestamp_audit[order_items_timestamp_audit['shipping_limit_date'] > '2018-12-31']
shipping_limit_after_2018

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
8643,13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,69.99,14.66
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,75.99,14.70
85729,c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44
85730,c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44


In [92]:
#whether these four records are genuine future dates or clear timestamp errors so verify with orders table
shipping_limit_order_id = shipping_limit_after_2018['order_id'].unique()
shipping_limit_order_id

<StringArray>
['13bdf405f961a6deec817d817f5c6624', '9c94a4ea2f7876660fa6f1b59b69c8e6',
 'c2bb89b5c1dd978d507284be78a04cb2']
Length: 3, dtype: str

In [93]:
verify_orders = orders_timestamp_audit[orders_timestamp_audit['order_id'].isin(shipping_limit_order_id)]
verify_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
36034,13bdf405f961a6deec817d817f5c6624,b279a1d441c73c1974d7a63618855aa0,canceled,2017-03-16 02:30:51,2017-03-16 02:30:51,NaT,NaT,2017-08-08
55368,9c94a4ea2f7876660fa6f1b59b69c8e6,22e922696a7d1ab9a19c6b702fedc387,shipped,2017-03-14 19:23:22,2017-03-14 19:23:22,2017-03-16 14:31:15,NaT,2017-08-04
86444,c2bb89b5c1dd978d507284be78a04cb2,6357fffb5704244d552615bbfcea1442,delivered,2017-05-23 22:28:36,2017-05-24 22:35:08,2017-05-29 02:03:28,2017-06-09 13:35:54,2017-10-11


#### Shipping-Limit Date Findings and Treatment Decision

`shipping_limit_date` was successfully converted from string to datetime, and the missing-value count remained zero before and after conversion. Therefore, no invalid timestamp formats were introduced.

The converted dates range from September 19, 2016, to April 9, 2020. Four item records across three orders contain shipping-limit dates after 2018. All four records belong to the same seller.

The corresponding Orders-table records show that all three orders were purchased in 2017 and have `canceled`, `shipped`, or `delivered` statuses. Therefore, the 2020 shipping-limit dates are inconsistent with the actual order lifecycle and are classified as seller-concentrated timestamp exceptions rather than genuine future transactions.

The affected item records will remain available for analyses that do not depend on the shipping deadline. However, they will be flagged and excluded from calculations involving seller shipping limits or shipping-deadline compliance. The incorrect dates will not be imputed because the correct deadlines cannot be reliably inferred from the available data.


## 4. Payments Table Audit

The Payments table contains payment records associated with customer orders. One order may contain multiple payment records when more than one payment method or payment sequence is used.

This audit evaluates the table’s structure, completeness, composite-key uniqueness, payment sequencing, payment-method categories, installment values, and monetary-value consistency before the data is used for revenue and payment analysis.

### 4.1 Dataset Shape and Structure

This subsection examines the table’s dimensions, columns, data types, and record grain to understand how payment records are represented for each order.


In [94]:
payments = pd.read_csv(Raw_Data_Path/"olist_order_payments_dataset.csv")
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [95]:
payments.shape

(103886, 5)

In [96]:
payments.columns

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')

In [97]:
payments.dtypes

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

In [98]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


**Initial observation:** The Payments table contains 103,886 payment records and 5 columns. The number of payment records is greater than the number of orders because one order may use multiple payment methods or contain multiple sequential payment records.

`order_id` and `payment_type` are stored as strings, `payment_sequential` and `payment_installments` are stored as integers, and `payment_value` is stored as a floating-point number. These data types are appropriate for the current audit.

The combination of `order_id` and `payment_sequential` is expected to uniquely identify each payment record, but this must be validated before treating it as a composite primary key.


### 4.2 Missing-Value Validation

Missing-value counts and percentages will be calculated for every Payments-table column to confirm whether the order identifier, payment sequence, payment type, installment count, and payment value are complete.


In [99]:
#payments missing data summary 
payments_missing_counts = payments.isna().sum()

payments_missing_precentage = (payments_missing_counts/len(payments))*100

payments_missing_summary = pd.DataFrame({"missing_count": payments_missing_counts, "missing_percentage": payments_missing_precentage})
payments_missing_summary

,missing_count,missing_percentage
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


**Finding:** No missing values were identified in any Payments-table column. Every payment record contains an order identifier, payment sequence, payment type, installment count, and payment value. Therefore, no missing-value treatment is required.


### 4.3 Composite-Key Validation

`order_id` may repeat because an order can contain multiple payment records. `payment_sequential` identifies the position of each payment record within an order. The combination of `order_id` and `payment_sequential` will be tested for duplicate combinations to determine whether it forms a valid composite primary key.

In [100]:
#Count distinct order_id–payment_sequential combinations
distinct_orderid_paymentsequential = len(payments[['order_id', 'payment_sequential']].drop_duplicates())
distinct_orderid_paymentsequential


103886

In [101]:
#Compare that count with the total number of rows
if distinct_orderid_paymentsequential == len(payments):
    print(f"Both values are same:{distinct_orderid_paymentsequential}")
else:
    print(f"both values are different")
    print(f"distinct order_id-sequential pair count: {distinct_orderid_paymentsequential}")
    print(f"total count of order_id's in payment table: {len(payments)}")

Both values are same:103886


In [102]:
#Count duplicated combinations directly in the original Payments table
int(payments[['order_id','payment_sequential']].duplicated().sum())

0

#### Composite-Key Findings

The Payments table contains 103,886 rows and 103,886 distinct `order_id`–`payment_sequential` combinations. No duplicated combinations or missing key values were identified.

Therefore, `order_id` and `payment_sequential` together form a valid composite primary key. `order_id` may repeat across multiple payment records, while `payment_sequential` distinguishes those records within each order.


### 4.4 Payment-Sequence Validation

`payment_sequential` is expected to begin at 1 and increase sequentially within each order. This check will identify non-positive sequence values and orders containing gaps or inconsistencies in their payment-record sequence.


In [103]:
#Minimum and maximum payment_sequential
print(f"minimum payment_sequential: {min(payments['payment_sequential'])}")
print(f"maximum payment_sequential: {max(payments['payment_sequential'])}")

minimum payment_sequential: 1
maximum payment_sequential: 29


In [104]:
#Number of records where it is less than or equal to zero
len(payments[payments['payment_sequential']<=0])

0

In [105]:
#Whether each order’s maximum payment sequence equals its number of payment records
pament_count_per_order = payments.groupby('order_id')['payment_sequential'].count()
max_pament_count_per_order = payments.groupby('order_id')['payment_sequential'].max()

payments_sequence_missmatch = pament_count_per_order != max_pament_count_per_order
int(payments_sequence_missmatch.sum())



80

In [106]:
mismatched_orders  = payments_sequence_missmatch[payments_sequence_missmatch==True]
mismatched_orders

order_id
00ac05fe0fc047c54418098eb64e3aaa    True
056c68d093c100017aab1f00f260705c    True
0668d086b3ae41adde3aed3dacbc8fae    True
0a7d898c6305101e69e9f5a05cd0130d    True
0d0e88a418636d40ea780339701db49c    True
                                    ... 
f0f94b7c7548150f33f5d9e7597d396f    True
f17a58e669a7f4cf7a7f227899f8abf7    True
f5691c2b1ca263490374d13d020bd950    True
fce3db7092e1f432d0ec790414b6a5e4    True
fe49c4a1416840f8ed6bd36daac85aae    True
Name: payment_sequential, Length: 80, dtype: bool

In [107]:
mismatched_orders_ids = mismatched_orders.index
len(mismatched_orders_ids)

80

In [108]:
mismatched_payment_records = payments[payments['order_id'].isin(mismatched_orders_ids)]
mismatched_payment_records[['order_id', 'payment_sequential', 'payment_type', 'payment_value']].head()

,order_id,payment_sequential,payment_type,payment_value
625,900fac51d850dfc38fceb9c413441c67,2,debit_card,98.33
2191,510bff1cf06be1143d3b6698df2fd486,2,credit_card,406.38
3071,d7d9f1ff0b6aa62f2d4e7fc99e60e8ea,2,debit_card,35.00
3272,cfa1591318ed6c901b0c80debfd4b811,2,credit_card,330.11
4218,e17ed9c20d79f29256a4a514e8d07230,2,debit_card,118.43


In [109]:
#Total number of rows in mismatched_payment_records
len(mismatched_payment_records)

82

In [110]:
#Frequency of each payment_sequential value within mismatched_payment_records

payment_seq_frequency = mismatched_payment_records['payment_sequential'].value_counts()
payment_seq_frequency

payment_sequential
2    80
3     2
Name: count, dtype: int64

In [111]:
#calculate the payment_type frequency within mismatched_payment_records
payment_type_frequency = mismatched_payment_records['payment_type'].value_counts()
payment_type_frequency

payment_type
debit_card     52
credit_card    29
boleto          1
Name: count, dtype: int64

In [112]:
payment_records_per_order = (mismatched_payment_records.groupby('order_id').size()).value_counts()
payment_records_per_order

1    78
2     2
Name: count, dtype: int64

#### Payment-Sequence Findings and Treatment Decision

`payment_sequential` values range from 1 to 29, and no zero or negative sequence values were identified. However, 80 orders contain sequence inconsistencies.

These 80 orders contain 82 payment records. Every affected order contains a record numbered 2, while two orders also contain a record numbered 3. None of the affected orders contains sequence 1. A total of 78 orders contain only one recorded payment, while 2 orders contain two recorded payments.

The affected records consist of 52 debit-card payments, 29 credit-card payments, and 1 boleto payment. Therefore, the issue is concentrated among card payments but is not limited to one payment type.

The dataset does not provide enough information to determine whether the sequence-1 payment records are missing or whether these orders were incorrectly numbered beginning with sequence 2. The records will not be deleted or renumbered during cleaning. Instead, they will be flagged and excluded from analyses requiring a complete payment sequence until their payment totals are evaluated during cross-table reconciliation.


### 4.5 Payment-Type Validation

The `payment_type` field will be examined for distinct categories, frequency distribution, blank values, and unexpected labels before it is used for payment-method analysis.


In [113]:
#Number of distinct payment types
distinct_payment_types = payments['payment_type'].nunique()
distinct_payment_types

5

In [114]:
#Frequency of each payment type
payment_type_frequency = payments['payment_type'].value_counts()
payment_type_frequency

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [115]:
#Number of blank or whitespace-only values
payment_type_blank_spaces = len(payments[payments['payment_type'].str.strip() == ''])
payment_type_blank_spaces

0

In [116]:
#Records with not_defined payment method
not_defined_payments = payments[payments['payment_type'] == "not_defined"]
not_defined_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [117]:
#inspect the corresponding three orders in orders_timestamp_audit using their order_id values
orders_timestamp_audit[orders_timestamp_audit['order_id'].isin(not_defined_payments['order_id'])]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaT,NaT,NaT,2018-09-12
39919,4637ca194b6387e2d538dc89b124b0ee,a73c1f73f5772cf801434bf984b0b1a7,canceled,2018-09-03 14:14:25,NaT,NaT,NaT,2018-09-10
40235,c8c528189310eaa44a745b8d9d26908b,197a2a6a77da93f678ea0d379f21da0a,canceled,2018-08-28 20:05:14,NaT,NaT,NaT,2018-09-11


#### Payment-Type Findings and Treatment Decision

The Payments table contains five payment-type categories: `credit_card`, `boleto`, `voucher`, `debit_card`, and `not_defined`. No blank or whitespace-only payment-type values were identified.

Credit cards are the most frequently used payment method, followed by boleto, voucher, and debit card. Only three records are labelled `not_defined`. All three have a payment value of zero and belong to cancelled orders that were never approved, shipped, or delivered.

Therefore, the `not_defined` records are considered lifecycle-consistent placeholder records rather than completed payments. They will remain in the raw data but will be excluded from delivered-order revenue and payment-method performance analysis.


### 4.6 Payment-Installment Validation

The `payment_installments` field will be examined for its numeric range, frequency distribution, and non-positive values. A valid completed payment is generally expected to contain at least one installment, so zero or negative values require further investigation.


In [118]:
#Minimum and maximum payment_installments
print(f"min payment_installment: {min(payments['payment_installments'])}")
print(f"max payment_installment: {max(payments['payment_installments'])}")

min payment_installment: 0
max payment_installment: 24


In [119]:
#Frequency of each installment count
payment_installments_frequency = payments['payment_installments'].value_counts()
payment_installments_frequency

payment_installments
1     52546
2     12413
3     10461
4      7098
10     5328
5      5239
8      4268
6      3920
7      1626
9       644
12      133
15       74
18       27
11       23
24       18
20       17
13       16
14       15
17        8
16        5
21        3
0         2
22        1
23        1
Name: count, dtype: int64

In [120]:
#Number of zero or negative installment values
print(f"Number of zero or negative installment values: {len(payments.loc[payments['payment_installments'] <= 0])}")

Number of zero or negative installment values: 2


In [121]:
zero_installment_payments = payments.loc[payments['payment_installments'] <= 0]
zero_installment_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [122]:
#verifying all payment records associated with order_id's of zero_installment_payments
payments_records_pi_2 = payments[payments['order_id'].isin(zero_installment_payments['order_id'])]
payments_records_pi_2

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [123]:
#Inspect in orders_timestamp_audit table for order_status 
zero_installment_payments_orderstatus = orders_timestamp_audit[orders_timestamp_audit['order_id'].isin(zero_installment_payments['order_id'])]
zero_installment_payments_orderstatus

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
63782,744bade1fcf9ff3f31d860ace076d422,5e5794daaa13f73e2f1cdb4114529843,delivered,2018-04-22 11:34:42,2018-04-24 19:04:46,2018-04-24 03:14:34,2018-04-27 20:55:28,2018-05-16
66368,1a57108394169c0b47d8f876acc9ba2d,48ebb06cf56dba9d009230cc751bb195,delivered,2018-05-15 16:25:14,2018-05-15 16:36:52,2018-05-17 12:37:00,2018-05-24 15:45:41,2018-06-06


#### Payment-Installment Findings and Treatment Decision

`payment_installments` ranges from 0 to 24. Most payment records contain one installment, while higher installment counts are primarily associated with credit-card transactions and are considered plausible.

Two credit-card records contain zero installments despite having positive payment values of 58.69 and 129.94. Both belong to delivered orders and are the only payment records available for their respective orders. They are also numbered as payment sequence 2, so they overlap with the previously identified sequence exceptions.

These payment values will be retained because they represent completed transactions. However, the two records will be flagged and excluded from analyses involving installment behavior. Their installment counts will not be imputed because the correct values cannot be determined reliably.


### 4.7 Payment-Value Validation

The `payment_value` column will be examined for its numeric distribution, negative values, zero values, and unusually high transactions. Zero-value payments will be investigated using their payment types and order statuses before a treatment decision is made.


In [124]:
#descriptive statistics for payment_value
payments['payment_value'].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [125]:
#Count negative values
len(payments[payments['payment_value'] < 0])

0

In [126]:
#Count zero values
len(payments[payments['payment_value'] == 0])

9

In [127]:
#Inspect the five highest payment values
top_5_payment_values = payments.sort_values(by='payment_value', ascending=False)
top_5_payment_values.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66


In [128]:
#zero value payment investigation
zero_value_payments_pay_val = payments[payments['payment_value'] == 0]
zero_value_payments_pay_val

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [129]:
#Select the order IDs for the zero-value voucher records.
zero_value_payments_pay_val[zero_value_payments_pay_val['payment_type'] == "voucher"]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [130]:
#Display all payment records associated with those order IDs
all_payments_associated = payments[payments['order_id'].isin(zero_value_payments_pay_val['order_id'])]
all_payments_associated

,order_id,payment_sequential,payment_type,payment_installments,payment_value
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
5163,8bcbe01d44d147f901cd3192671144db,3,voucher,1,6.25
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
20963,8bcbe01d44d147f901cd3192671144db,1,credit_card,1,36.21
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51


#### Payment-Value Findings and Treatment Decision

`payment_value` ranges from 0.00 to 13,664.08, with a median of 100.00 and a mean of 154.10. No negative payment values were identified. The highest payment values correspond to plausible high-value transactions, and several closely match the expensive item and freight totals observed in the Order Items table.

Nine zero-value payment records were identified. Three are `not_defined` placeholder records associated with cancelled orders. The remaining six are voucher records across five orders. Every affected voucher order contains at least one additional positive payment record, indicating that the zero-value entries do not represent the order’s entire payment history.

The zero-value records will remain in the raw dataset, but they will be excluded from payment-method usage counts and payment-value analysis because they do not represent a positive monetary contribution. They do not affect order-level payment totals because their value is zero. High-value payments will be retained and evaluated later during cross-table payment reconciliation.


## 5. Reviews Table Audit

The Reviews table contains customer feedback associated with orders. Each record includes a review identifier, related order, review score, optional written feedback, review-creation date, and response timestamp.

This audit evaluates the table’s structure, missing values, identifier uniqueness, duplicate records, review-score validity, optional comment fields, and timestamp consistency before the data is used for customer-satisfaction analysis.

### 5.1 Dataset Shape and Structure

This subsection examines the table’s dimensions, columns, data types, and expected record grain to understand how reviews are associated with customer orders.


In [131]:
reviews = pd.read_csv(Raw_Data_Path/'olist_order_reviews_dataset.csv')
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [132]:
reviews.shape

(99224, 7)

In [133]:
reviews.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')

In [134]:
reviews.dtypes

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

In [135]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


**Initial observation:** The Reviews table contains 99,224 review records and 7 columns. `review_id` and `order_id` are stored as strings, while `review_score` is stored as an integer, which is appropriate for score validation.

`review_creation_date` and `review_answer_timestamp` are currently stored as strings and should be converted to datetime during preparation.

The review-title and review-message fields contain substantial missing values. However, written feedback is optional, so these missing values should not automatically be classified as data-quality errors. Their frequency and relationship with review scores will be investigated before determining treatment.


### 5.2 Missing-Value Validation

Missing-value counts and percentages will be calculated for every Reviews-table column. Missing identifiers, scores, or timestamps would represent data-quality concerns, while missing review titles and messages may represent customers who submitted only a numeric rating.


In [136]:
reviews_missing_count = reviews.isna().sum()

reviews_missing_percentage = (reviews_missing_count/len(reviews))*100

reviews_missing_value_summary = pd.DataFrame({"reviews_missing_count":reviews_missing_count, "reviews_missing_percentage":reviews_missing_percentage})
reviews_missing_value_summary

,reviews_missing_count,reviews_missing_percentage
review_id,0,0.000000
order_id,0,0.000000
review_score,0,0.000000
review_comment_title,87656,88.341530
review_comment_message,58247,58.702532
review_creation_date,0,0.000000
review_answer_timestamp,0,0.000000


In [137]:
#Review-score distribution among records with a missing title
reviews_missing_title = reviews[reviews['review_comment_title'].isna()]
frequency_reviews_missing_title = reviews_missing_title['review_score'].value_counts()
frequency_reviews_missing_title

review_score
5    50670
4    17407
1     9551
3     7355
2     2673
Name: count, dtype: int64

In [138]:
#Review-score distribution among records with a missing message
reviews_missing_message = reviews[reviews['review_comment_message'].isna()]
frequency_reviews_missing_message = reviews_missing_message['review_score'].value_counts()
frequency_reviews_missing_message

review_score
5    36774
4    13166
3     4622
1     2679
2     1006
Name: count, dtype: int64

In [139]:
#Number of reviews where both title and message are missing
len(reviews[(reviews['review_comment_title'].isna()) & (reviews['review_comment_message'].isna())])

56518

#### Missing-Value Findings and Treatment Decision

No missing values were identified in `review_id`, `order_id`, `review_score`, `review_creation_date`, or `review_answer_timestamp`.

However, 87,656 review titles (88.34%) and 58,247 review messages (58.70%) are missing. A total of 56,518 reviews contain neither a title nor a message. Missing written feedback occurs across all five review-score levels, confirming that customers can submit a numeric rating without providing text.

These missing text values are considered expected optional-field omissions rather than data-quality errors. They will not be imputed or replaced with artificial text. `review_score` can still be used for satisfaction analysis, while separate comment-availability indicators may be created later for text-based analysis.


### 5.3 Identifier and Duplicate Validation

The uniqueness of `review_id` and `order_id` will be examined to determine whether reviews contain repeated identifiers or multiple records for the same order. Exact duplicate rows will also be checked before selecting an appropriate review-level grain for analysis.


In [140]:
#Number of distinct review_id values and duplicated occurrences
distinct_review_id = reviews['review_id'].nunique()
distinct_review_id

98410

In [141]:
#Number of distinct order_id values and duplicated occurrences
distinct_review_order_id = reviews['order_id'].nunique()
distinct_review_order_id

98673

In [142]:
#Number of exact duplicate rows
reviews_duplicate_rows = int(reviews.duplicated().sum())
reviews_duplicate_rows

0

In [143]:
#Duplicated review_id occurrences
int(reviews['review_id'].duplicated().sum())

814

In [144]:
#Duplicated order_id occurrences
int(reviews['order_id'].duplicated().sum())

551

In [145]:
#same review_id–order_id combination
int(reviews[['review_id','order_id']].duplicated().sum())

0

In [146]:
#Number of review_id values associated with more than one distinct order_id
review_id_with_order_ids = reviews.groupby('review_id')['order_id'].nunique()
review_id_with_order_ids_gr1 = review_id_with_order_ids[review_id_with_order_ids>1]
len(review_id_with_order_ids_gr1)

789

In [147]:
#maximum number of distinct orders associated with a single review_id
max(review_id_with_order_ids)

3

In [148]:
review_id_with_order_ids_gr1.value_counts()

order_id
2    764
3     25
Name: count, dtype: int64

In [149]:
#Group the records by order_id
order_id_with_review_ids = reviews.groupby('order_id')['review_id'].nunique()
order_id_with_review_ids_gr1 = order_id_with_review_ids[order_id_with_review_ids>1]
len(order_id_with_review_ids_gr1)

547

In [150]:
#maximum number of distinct review IDs associated with one order
max(order_id_with_review_ids)

3

In [151]:
order_id_with_review_ids_gr1.value_counts()

review_id
2    543
3      4
Name: count, dtype: int64

#### Identifier and Duplicate Findings

The Reviews table contains **99,224 records**, with **98,410 distinct `review_id` values** and **98,673 distinct `order_id` values**. No exact duplicate rows or duplicated `review_id`–`order_id` combinations were found.

However, neither identifier is independently unique:

* `review_id` contains **814 duplicated occurrences**. A total of **789 review IDs** are associated with multiple orders: 764 are linked to 2 orders and 25 are linked to 3 orders.
* `order_id` contains **551 duplicated occurrences**. A total of **547 orders** are associated with multiple review IDs: 543 have 2 review IDs and 4 have 3 review IDs.
* The maximum relationship in either direction is 3 records.

Therefore, neither `review_id` nor `order_id` should be treated as a standalone primary key. The combination of `review_id` and `order_id` uniquely identifies each record in the raw Reviews table.

No records will be removed during the audit because the repeated identifiers are not exact duplicate rows. Before creating an order-level review dataset, the repeated-order records should be investigated using their review scores and timestamps. A documented aggregation or record-selection rule will then be used to prevent orders with multiple reviews from receiving additional weight in order-level analysis.


### 5.4 Review-Score Validation

The `review_score` column will be examined to confirm that all values fall within the expected rating scale and to understand the distribution of customer ratings. Missing, invalid, and out-of-range scores will also be checked before the column is used in customer-satisfaction analysis.


In [152]:
#The distinct review_score values
distinct_review_score = reviews['review_score'].unique()
distinct_review_score

array([4, 5, 1, 3, 2])

In [153]:
#The frequency of each score.
review_score_frequency = reviews['review_score'].value_counts()
review_score_frequency

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [154]:
#The minimum and maximum review_score.

print(f"min review_score: {min(reviews['review_score'])}")
print(f"max review_score: {max(reviews['review_score'])}")

min review_score: 1
max review_score: 5


In [155]:
#The number of records where review_score is below 1 or above 5.
review_score_l1_g5 = len(reviews.loc[(reviews['review_score']<1) | (reviews['review_score']>5)])
review_score_l1_g5

0

In [156]:
#The descriptive statistics of review_score, especially its mean and median.
reviews['review_score'].describe()

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

#### Review-Score Findings

The `review_score` column contains **99,224 non-null values** and uses the complete expected rating scale from 1 to 5. No scores below 1 or above 5 were found.

The score distribution is:

* Score 5: 57,328 reviews
* Score 4: 19,142 reviews
* Score 3: 8,179 reviews
* Score 2: 3,151 reviews
* Score 1: 11,424 reviews

The mean review score is approximately **4.09**, while the median is **5**. The first quartile is 4 and the third quartile is 5, showing that customer ratings are concentrated toward the positive end of the scale.

The column requires no cleaning and can be retained for customer-satisfaction and product-performance analysis.


### 5.5 Review Timestamp Validation

The `review_creation_date` and `review_answer_timestamp` columns are currently stored as strings. They will be converted to datetime format, checked for parsing errors, and examined for inconsistent chronological sequences before being used in time-based review analysis.


In [157]:
reviews_timestamp_audit = pd.read_csv(Raw_Data_Path/'olist_order_reviews_dataset.csv')
reviews_timestamp_audit.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [158]:
reviews_timestamp_audit.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [159]:
#convert review_creation_date and review_creation_date to datetime format
dt_cols = ['review_creation_date', 'review_answer_timestamp']
for col in dt_cols:
    reviews_timestamp_audit[col] = pd.to_datetime(reviews_timestamp_audit[col], errors='coerce')

reviews_timestamp_audit.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  str           
 4   review_comment_message   40977 non-null  str           
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 5.3 MB


In [160]:
#Count missing values in the two timestamp columns in the original reviews DataFrame
creation_date_null_values = int(reviews['review_creation_date'].isna().sum())
print(f"creation_date nulls: {creation_date_null_values}")

answer_timestamp_null_values = int(reviews['review_answer_timestamp'].isna().sum())
print(f"answer_timestamp nulls: {answer_timestamp_null_values}")



creation_date nulls: 0


answer_timestamp nulls: 0


In [161]:
audit_creation_date_null_values = int(reviews_timestamp_audit['review_creation_date'].isna().sum())
print(f"audit_creation_date nulls: {audit_creation_date_null_values}")

audit_answer_timestamp_null_values = int(reviews_timestamp_audit['review_answer_timestamp'].isna().sum())
print(f"audit_answer_date nulls: {audit_answer_timestamp_null_values}")

audit_creation_date nulls: 0
audit_answer_date nulls: 0


In [162]:
reviews_timestamp_audit

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01,2018-07-02 12:59:13


In [163]:
#review_answer_timestamp is earlier than review_creation_date

answer_timestamp_before_creation = len(reviews_timestamp_audit[reviews_timestamp_audit['review_answer_timestamp'] < reviews_timestamp_audit['review_creation_date']])
answer_timestamp_before_creation 

0

In [164]:
#minimum and maximum values for review_creation_date
print(f"min creation_date: {min(reviews_timestamp_audit['review_creation_date'])}")
print(f"max creation_date: {max(reviews_timestamp_audit['review_creation_date'])}")

min creation_date: 2016-10-02 00:00:00
max creation_date: 2018-08-31 00:00:00


In [165]:
#minimum and maximum values for review_answer_timestamp
print(f"min answer_timestamp: {min(reviews_timestamp_audit['review_answer_timestamp'])}")
print(f"max answer_timestamp: {max(reviews_timestamp_audit['review_answer_timestamp'])}")

min answer_timestamp: 2016-10-07 18:32:28
max answer_timestamp: 2018-10-29 12:27:35


In [166]:
#Create a new duration column

reviews_timestamp_audit['duration'] = reviews_timestamp_audit['review_answer_timestamp'] - reviews_timestamp_audit['review_creation_date']
reviews_timestamp_audit.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,duration
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59,0 days 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13,1 days 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24,1 days 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,0 days 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53,1 days 10:26:53


In [167]:
reviews_timestamp_audit['duration'].describe()

count                     99224
mean     3 days 03:34:33.029700
std      9 days 21:21:40.258026
min             0 days 02:08:29
25%      1 days 00:07:00.750000
50%      1 days 16:11:55.500000
75%             3 days 02:29:08
max           518 days 16:46:52
Name: duration, dtype: object

In [168]:
#Count records with a duration greater than 30 days

duration_grt30d= reviews_timestamp_audit[reviews_timestamp_audit['duration'] > pd.Timedelta(days=30)]
len(duration_grt30d)

682

In [169]:
#Display the five records with the longest duration
reviews_with_longer_duration = reviews_timestamp_audit.sort_values(by='duration', ascending=False)
reviews_with_longer_duration.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,duration
41620,40dad6438b6cbec46d936bec2377778c,bb5849f8ba21da43ffa31ea52ba81b37,1,NaN,NaN,2017-03-24,2018-08-24 16:46:52,518 days 16:46:52
66672,eeb94b15f672afaf38e8d74b6ddb25f8,a8a2941e92152e0edc39bb862cd6b6e2,5,NaN,NaN,2017-02-14,2018-07-11 15:47:37,512 days 15:47:37
22059,083e71bf744a3219a07a948f12243f9a,898ded7ecb1e4d92997856bd8b7a8aa1,5,NaN,NaN,2017-03-28,2018-08-18 01:36:09,508 days 01:36:09
8758,cc2b2638ac2654471bb4f652a1a965fa,3d80d12c0beadfcdcf2b4df407112f3f,4,NaN,NaN,2017-05-17,2018-08-31 16:19:53,471 days 16:19:53
13920,3638b013e70d3e74fa51cb4fce6fd904,3b39d2ddd9066f382c924cc2c51dc584,5,NaN,otimo,2017-06-14,2018-09-03 20:51:34,446 days 20:51:34


In [170]:
duration_grt30d_percentage = (len(duration_grt30d)/len(reviews_timestamp_audit))*100
duration_grt30d_percentage

0.6873337095863904

#### Review Timestamp Findings

The `review_creation_date` and `review_answer_timestamp` columns were successfully converted from strings to datetime format. The missing-value counts remained zero before and after conversion, confirming that no invalid timestamp strings were converted to `NaT`.

The timestamp ranges are:

* Review creation dates: 2016-10-02 to 2018-08-31
* Review answer timestamps: 2016-10-07 to 2018-10-29

No records were found where `review_answer_timestamp` occurred before `review_creation_date`, confirming that the internal review timestamp sequence is consistent.

The median response duration is approximately **1 day and 16 hours**, while the mean is approximately **3 days and 3 hours**. Seventy-five percent of reviews were answered within approximately **3 days and 2 hours**. However, the maximum response duration is approximately **518 days and 17 hours**, indicating the presence of unusually long response times.

A total of **682 reviews (0.69%)** have response durations greater than 30 days. These records are rare and remain chronologically valid, so they will not be removed or modified. They may be flagged and examined separately or excluded from typical response-time calculations because they could disproportionately affect the mean. Their review scores and comments can remain available for customer-satisfaction analysis.


## 6. Products Table Audit

The Products table will be audited to validate its structure, missing values, product identifier uniqueness, category-name completeness, and the quality of its numeric attributes. Product measurements such as weight, dimensions, description length, and photo quantity will be checked for invalid, zero, or unusually large values before they are used in product- and category-level analysis.

### 6.1 Dataset Shape and Structure

The table’s dimensions, column names, data types, and non-null counts will first be inspected to understand its structure and identify fields requiring further validation.


In [171]:
products = pd.read_csv(Raw_Data_Path/'olist_products_dataset.csv')
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [172]:
products.shape

(32951, 9)

In [173]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [174]:
products.dtypes

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [175]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


#### Initial Structure Observations

The Products table contains **32,951 rows and 9 columns**, with each row expected to represent one product. The uniqueness of `product_id` has not yet been validated.

The table contains product identifiers, category names, product-content attributes, weight, and physical dimensions. The `product_id` and `product_category_name` columns are stored as strings, while the remaining seven attributes are stored as numeric values.

The source dataset uses the misspelled column names `product_name_lenght` and `product_description_lenght`. These names should be documented and may be corrected during cleaning.

The structure inspection indicates potential missing values in the category, content, weight, and dimension columns. These will be quantified before deciding how they should be treated.

### 6.2 Missing-Value Validation

Missing values will be counted and expressed as a percentage of the Products table. The affected columns will then be investigated together to determine whether the missing product attributes follow a common pattern.


In [176]:
#missing values summary
products_missing_count = products.isna().sum()
products_missing_percentage = (products_missing_count/len(products)) * 100

products_missing_summary = pd.DataFrame({"missing_count":products_missing_count, "missing_percentage":products_missing_percentage})
products_missing_summary

,missing_count,missing_percentage
product_id,0,0.000000
product_category_name,610,1.851234
product_name_lenght,610,1.851234
product_description_lenght,610,1.851234
product_photos_qty,610,1.851234
product_weight_g,2,0.006070
product_length_cm,2,0.006070
product_height_cm,2,0.006070
product_width_cm,2,0.006070


In [177]:
#Check whether the same 610 rows are missing category, name length, description length, and photo quantity together

products_missing_table_1 = products[products['product_category_name'].isna() | products['product_name_lenght'].isna() | products['product_description_lenght'].isna() | products['product_photos_qty'].isna() ]
len(products_missing_table_1)

610

In [178]:
#Check whether the same 2 rows are missing product_weight, product_length, product_height, and product_width together
products_missing_table_2 = products[products['product_weight_g'].isna() | products['product_length_cm'].isna() | products['product_height_cm'].isna() | products['product_width_cm'].isna() ]
len(products_missing_table_2)

2

In [179]:
#the 2 values missing product present in same 610 products 
products_missing_1_2 = products_missing_table_1[products_missing_table_1['product_id'].isin(products_missing_table_2['product_id'])]
len(products_missing_1_2)

1

In [180]:
products_missing_table_2

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [181]:
#Determine how many of those 610 product IDs appear in the Order Items table
products_in_orderitems = order_items[order_items['product_id'].isin(products_missing_table_1['product_id'])]
products_in_orderitems['order_id'].nunique()


1451

In [182]:
products_in_orderitems['product_id'].nunique()

610

In [183]:
len(products_in_orderitems['product_id'])

1603

In [184]:
#what percentage the 1,603 affected item records represent out of all Order Items records.
total_affected_percentage = (len(products_in_orderitems['product_id'])/len(order_items))*100
total_affected_percentage

1.4229915667998225

In [185]:
ot_missing_product_per_order = order_items[order_items['product_id'].isin(products_missing_table_2['product_id'])]
len(ot_missing_product_per_order)


18

In [186]:
ot_missing_product_per_order['order_id'].nunique()

16

#### Missing-Value Findings and Treatment Decision

The `product_id` column contains no missing values. However, four product-content columns—`product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty`—each contain **610 missing values (1.85%)**. Investigation confirmed that all four attributes are missing together for the same 610 products.

All 610 affected products appear in the Order Items table, representing **1,603 item records across 1,451 orders**. These records account for approximately **1.42% of all order-item records**. Therefore, removing them would undercount actual transactions. The products will be retained, and missing categories may be represented as `unknown` or `unclassified` in the cleaned analytical dataset. Missing content attributes will not be arbitrarily imputed.

The weight and three dimension columns each contain **2 missing values (0.006%)**, occurring together in the same two products. These products are associated with **18 item records across 16 orders**. One product has complete category and content information but missing measurements, while the other is also included among the 610 products with missing content information.

Both products will be retained for sales analysis because their identifiers and transaction records remain valid. They will be excluded only from calculations that specifically require product weight or dimensions.

### 6.3 Product Identifier Validation

The `product_id` column will be evaluated for missing values, uniqueness, and duplicated occurrences to determine whether it can serve as the primary key of the Products table.


In [187]:
#compare the total number of product rows with the number of distinct product IDs, then count duplicated product IDs
num_product_rows = len(products)
num_distinct_product_id = products['product_id'].nunique()
num_duplicated_product_id = int(products['product_id'].duplicated().sum())

print(f"num_product_rows: {num_product_rows}")
print(f"num_distinct_product_id: {num_distinct_product_id}")
print(f"num_duplicated_product_id: {num_duplicated_product_id}")

num_product_rows: 32951
num_distinct_product_id: 32951
num_duplicated_product_id: 0


#### Product Identifier Findings

The Products table contains **32,951 rows and 32,951 distinct `product_id` values**. No missing or duplicated product identifiers were found.

Therefore, `product_id` uniquely identifies each product and can serve as the primary key of the Products table. It can be used to establish the one-to-many relationship between Products and Order Items.

### 6.4 Numeric Attribute Validation

The product-content and physical-measurement attributes will be examined for invalid values and unusual ranges. This includes product name length, description length, photo quantity, weight, length, height, and width. Missing values were already documented and will be excluded automatically from the numeric summaries.


In [188]:
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [189]:
products['product_name_lenght'].describe()

count    32341.000000
mean        48.476949
std         10.245741
min          5.000000
25%         42.000000
50%         51.000000
75%         57.000000
max         76.000000
Name: product_name_lenght, dtype: float64

In [190]:
products['product_description_lenght'].describe()

count    32341.000000
mean       771.495285
std        635.115225
min          4.000000
25%        339.000000
50%        595.000000
75%        972.000000
max       3992.000000
Name: product_description_lenght, dtype: float64

In [191]:
products['product_photos_qty'].describe()

count    32341.000000
mean         2.188986
std          1.736766
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max         20.000000
Name: product_photos_qty, dtype: float64

In [192]:
products['product_weight_g'].describe()

count    32949.000000
mean      2276.472488
std       4282.038731
min          0.000000
25%        300.000000
50%        700.000000
75%       1900.000000
max      40425.000000
Name: product_weight_g, dtype: float64

In [193]:
products['product_length_cm'].describe()

count    32949.000000
mean        30.815078
std         16.914458
min          7.000000
25%         18.000000
50%         25.000000
75%         38.000000
max        105.000000
Name: product_length_cm, dtype: float64

In [194]:
products['product_height_cm'].describe()

count    32949.000000
mean        16.937661
std         13.637554
min          2.000000
25%          8.000000
50%         13.000000
75%         21.000000
max        105.000000
Name: product_height_cm, dtype: float64

In [195]:
products['product_width_cm'].describe()

count    32949.000000
mean        23.196728
std         12.079047
min          6.000000
25%         15.000000
50%         20.000000
75%         30.000000
max        118.000000
Name: product_width_cm, dtype: float64

In [196]:
#products with a weight equal to zero
zero_weight_products = products[products['product_weight_g']==0]
zero_weight_products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [197]:
len(zero_weight_products)

4

In [198]:
#How many Order Items records contain these four products.
ot_with_zero_weight_products = order_items[order_items['product_id'].isin(zero_weight_products['product_id'])]
ot_with_zero_weight_products

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
2972,06afc1144eb9f51ef2aa90ec9223c7f4,1,e673e90efa65a5409ff4196c038bb5af,b39d7fe263ef469605dbb32608aee0af,2018-08-23 17:25:20,129.9,23.71
2973,06afc1144eb9f51ef2aa90ec9223c7f4,2,e673e90efa65a5409ff4196c038bb5af,b39d7fe263ef469605dbb32608aee0af,2018-08-23 17:25:20,129.9,23.71
3052,06d9e69034388abf6da64378e10737b8,1,36ba42dd187055e1fbe943b2d11430ca,b39d7fe263ef469605dbb32608aee0af,2018-08-10 09:10:11,100.0,23.85
3053,06d9e69034388abf6da64378e10737b8,2,36ba42dd187055e1fbe943b2d11430ca,b39d7fe263ef469605dbb32608aee0af,2018-08-10 09:10:11,100.0,23.85
14080,200b121c28e10ef638131a7c76753327,1,81781c0fed9fe1ad6e8c81fca1e1cb08,b39d7fe263ef469605dbb32608aee0af,2018-08-14 16:10:16,100.0,19.89
31488,476b812a7e4fc972646eb390517bddcb,1,e673e90efa65a5409ff4196c038bb5af,b39d7fe263ef469605dbb32608aee0af,2018-08-22 11:30:42,129.9,23.71
32984,4abc7b5330425bcf9c2f7f48151a88c0,1,8038040ee2a71048d4bdbbdc985b69ab,b39d7fe263ef469605dbb32608aee0af,2018-08-09 21:31:33,129.9,14.49
79374,b489f7ae130ba3fd26b0a20f8cc81c61,1,e673e90efa65a5409ff4196c038bb5af,b39d7fe263ef469605dbb32608aee0af,2018-08-22 20:49:57,129.9,23.71


In [199]:
#How many distinct orders are affected.
ot_with_zero_weight_products['order_id'].nunique()

6

In [200]:
#inspect the five products with the highest recorded weight.
high_weight_products = products.sort_values(by='product_weight_g', ascending=False)
high_weight_products.head(5)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
25166,26644690fde745fc4654719c3904e1db,cama_mesa_banho,59.0,534.0,1.0,40425.0,13.0,65.0,28.0
11327,8d4e92265a16e69a1e1d76e67e46d72f,instrumentos_musicais,45.0,422.0,5.0,30000.0,80.0,75.0,36.0
31042,5459103ed3b2f852d267e6a70b00dc24,casa_construcao,56.0,706.0,1.0,30000.0,40.0,79.0,40.0
5783,b2b7d701c1f12d5a0f52ec90b0a8f819,beleza_saude,58.0,424.0,1.0,30000.0,55.0,75.0,61.0
27230,e5f2d52b802189ee658865ca93d83a8f,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0


In [201]:
#product records having the maximum length
max_Length_products = products.sort_values(by='product_length_cm', ascending=False)
max_Length_products.head(1)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
16685,87c070a9069fffd77616e3581d8d2ba1,esporte_lazer,36.0,213.0,1.0,5900.0,105.0,8.0,27.0


In [202]:
#product records having the maximum Height
max_Height_products = products.sort_values(by='product_height_cm', ascending=False)
max_Height_products.head(1)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
17371,995a6110a2705a9401669fb4cf939241,malas_acessorios,45.0,416.0,4.0,12700.0,18.0,105.0,40.0


In [203]:
#product records having the maximum Width
max_Width_products = products.sort_values(by='product_width_cm', ascending=False)
max_Width_products.head(1)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
26970,b17808303e15dd50538c011b44295427,cama_mesa_banho,56.0,502.0,3.0,1050.0,23.0,93.0,118.0


#### Numeric Attribute Findings and Treatment Decision

The product-content attributes contain only positive non-missing values:

* Product name length ranges from 5 to 76 characters.
* Product description length ranges from 4 to 3,992 characters.
* Product photo quantity ranges from 1 to 20.

No zero or negative values were found in these attributes.

Product weight ranges from 0 to 40,425 grams. Four products contain a weight of zero despite having non-zero physical dimensions. All four belong to `cama_mesa_banho`, share identical dimensions, and were sold by the same seller. They affect **8 order-item records across 6 orders**, indicating a small catalogue-data issue. The products will remain in sales analysis, but their zero weights will be treated as missing and excluded from weight-dependent calculations.

The highest recorded weights, between 30 and 40.425 kilograms, occur across categories capable of containing heavy or bundled products. Although unusual, there is insufficient evidence to classify them as errors, so they will be retained as high-weight outliers.

All non-missing length, height, and width values are positive. The maximum measurements—105 cm in length, 105 cm in height, and 118 cm in width—were inspected and found to be plausible based on their categories and other dimensions. No measurement outliers will be removed.

### 6.5 Product-Category Validation

The `product_category_name` column will be checked for distinct values, blank entries, inconsistent formatting, and category frequency. Its values will later be compared with the category-translation table to determine whether all Portuguese category names have an English translation.


In [204]:
#The number of distinct non-missing product categories
products['product_category_name'].nunique()

73

In [205]:
#category values that are empty or contain only whitespace
len(products[products['product_category_name'].str.strip() == ''])

0

In [206]:
#frequency of each non-missing product category
products['product_category_name'].value_counts()

product_category_name
cama_mesa_banho                  3029
esporte_lazer                    2867
moveis_decoracao                 2657
beleza_saude                     2444
utilidades_domesticas            2335
                                 ... 
fashion_roupa_infanto_juvenil       5
casa_conforto_2                     5
pc_gamer                            3
seguros_e_servicos                  2
cds_dvds_musicais                   1
Name: count, Length: 73, dtype: int64

#### Category Translation Coverage

The product-category translation table will be inspected to confirm that each Portuguese category name has a unique English translation. Missing, duplicated, and unmatched category mappings will be identified before translated category names are used in analysis.


In [207]:
translation = pd.read_csv(Raw_Data_Path/'product_category_name_translation.csv')
translation

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [208]:
translation.shape

(71, 2)

In [209]:
translation.columns

Index(['product_category_name', 'product_category_name_english'], dtype='str')

In [210]:
translation.dtypes

product_category_name            str
product_category_name_english    str
dtype: object

In [211]:
translation.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [212]:
#distinct Portuguese category names in the translation table
translation['product_category_name'].nunique()

71

In [213]:
#duplicated Portuguese category-name occurrences
int(translation['product_category_name'].duplicated().sum())

0

In [214]:
#blank or whitespace-only values in both translation columns
print(f"blank_portugues_name: {len(translation[translation['product_category_name'].str.strip() == ''])}")
print(f"blank_english_name: {len(translation[translation['product_category_name_english'].str.strip() == ''])}")

blank_portugues_name: 0
blank_english_name: 0


In [215]:
unmatched_product_names = products.loc[(products['product_category_name'].notna() ) & ~(products['product_category_name'].isin(translation['product_category_name'])), 'product_category_name'].drop_duplicates()
unmatched_product_names

1628                                         pc_gamer
5821    portateis_cozinha_e_preparadores_de_alimentos
Name: product_category_name, dtype: str

In [216]:
unmatched_records = products.loc[products['product_category_name'].isin(unmatched_product_names)]
unmatched_product_category_name_freq = unmatched_records['product_category_name'].value_counts()
unmatched_product_category_name_freq

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

In [217]:
#How many appear in the Order Items table
unmatched_pcn_records_ot = order_items[order_items['product_id'].isin(unmatched_records['product_id'])]
len(unmatched_pcn_records_ot)

24

In [218]:
#The number of distinct product_id values
unmatched_pcn_records_ot['product_id'].nunique()

13

In [219]:
#The number of distinct order_id values
unmatched_pcn_records_ot['order_id'].nunique()

22

#### Product-Category and Translation Findings

The Products table contains **73 distinct non-missing category names**. No blank or whitespace-only category values were found. Category frequency varies considerably, from 3,029 products in `cama_mesa_banho` to 1 product in `cds_dvds_musicais`.

The category-translation table contains **71 complete and unique Portuguese-to-English mappings**, with no missing, blank, or duplicated Portuguese category names.

Two Product categories are absent from the translation table:

* `portateis_cozinha_e_preparadores_de_alimentos`: 10 products
* `pc_gamer`: 3 products

All 13 affected products appear in Order Items, representing **24 item records across 22 orders**. Therefore, they should not be removed from category-level analysis.

During cleaning, documented English labels will be supplied manually:

* `pc_gamer` → `pc_gamer`
* `portateis_cozinha_e_preparadores_de_alimentos` → `small_appliances_kitchen_and_food_preparation`

The 610 products with missing category names will remain classified as `unknown` or `unclassified`. The original Portuguese values will be preserved for traceability.


## 7. Sellers Table Audit

The Sellers table will be audited to validate its structure, missing values, seller identifier uniqueness, and geographic attributes. Seller city, state, and ZIP-code prefixes will be checked for formatting and consistency before they are used in seller-location and regional analysis.

### 7.1 Dataset Shape and Structure

The number of records, column names, data types, and non-null counts will first be inspected to understand the table’s structure and identify fields requiring validation.


In [220]:
sellers = pd.read_csv(Raw_Data_Path/'olist_sellers_dataset.csv')
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [221]:
sellers.shape

(3095, 4)

In [222]:
sellers.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='str')

In [223]:
sellers.dtypes

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

In [224]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


#### Initial Structure Observations

The Sellers table contains **3,095 rows and 4 columns**, with each row expected to represent one seller. The uniqueness of `seller_id` has not yet been validated.

The table contains seller identifiers and geographic attributes. `seller_id`, `seller_city`, and `seller_state` are stored as strings, while `seller_zip_code_prefix` is stored as an integer. Because ZIP-code prefixes can begin with zero, the integer format may remove leading zeros and will require formatting review during cleaning.

The initial structure output shows complete non-null counts, but a formal missing-value summary will be created to confirm this.

### 7.2 Missing-Value Validation

Missing values will be counted and expressed as a percentage of all seller records to confirm whether each field is complete.


In [225]:
#missing-value count and percentage for every Sellers column.
sellers_missing_count = sellers.isna().sum()
sellers_missing_prececntage = (sellers_missing_count/len(sellers)) * 100

sellers_missing_summary = pd.DataFrame({"missing_count":sellers_missing_count , "missing_percentage":sellers_missing_prececntage})
sellers_missing_summary


,missing_count,missing_percentage
seller_id,0,0.0
seller_zip_code_prefix,0,0.0
seller_city,0,0.0
seller_state,0,0.0


#### Missing-Value Findings

The Sellers table contains no missing values. All 3,095 records have complete seller identifiers, ZIP-code prefixes, cities, and states. No missing-value treatment is required for this table.

### 7.3 Seller Identifier Validation

The `seller_id` column will be checked for uniqueness and duplicated occurrences to determine whether it can serve as the primary key of the Sellers table.


In [226]:
print(f"total number of seller rows: {len(sellers)}")
print(f"total number of distinct seller_id's: {sellers['seller_id'].nunique()}")
print(f"total number of duplicated seller_id's: {sellers['seller_id'].duplicated().sum()}")


total number of seller rows: 3095
total number of distinct seller_id's: 3095
total number of duplicated seller_id's: 0


#### Seller Identifier Findings

The Sellers table contains **3,095 rows and 3,095 distinct `seller_id` values**. No missing or duplicated seller identifiers were found.

Therefore, `seller_id` uniquely identifies each seller and can serve as the primary key of the Sellers table. It can be used to establish the one-to-many relationship between Sellers and Order Items.

### 7.4 Seller Geographic Validation

Seller state, city, and ZIP-code-prefix values will be examined for invalid formats, blank entries, inconsistent geographic mappings, and potential loss of leading zeros.


In [227]:
#number of distinct states
sellers['seller_state'].nunique()

23

In [228]:
#frequency of each state
sellers['seller_state'].value_counts()

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

In [229]:
#Blank or whitespace-only values
len(sellers[sellers['seller_state'].str.strip() ==''])

0

In [230]:
#Values that are not exactly two uppercase characters.
len(sellers.loc[ (sellers['seller_state'].str.len() !=2) | (~sellers['seller_state'].str.isupper())])

0

In [231]:
#The number of distinct city values
sellers['seller_city'].nunique()

611

In [232]:
#Blank or whitespace-only values.
len(sellers[sellers['seller_city'].str.strip() == ''])

0

In [233]:
#Values containing leading or trailing spaces.
sellers_city_with_leading_space = sellers['seller_city'].str.startswith(' ')
sellers_city_with_trailing_space = sellers['seller_city'].str.endswith(' ')

sellers_city_leading_r_trailing_space = sellers.loc[sellers_city_with_leading_space | sellers_city_with_trailing_space]
len(sellers_city_leading_r_trailing_space)


0

In [234]:
sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [235]:
#The number of distinct ZIP-code prefixes.
sellers['seller_zip_code_prefix'].nunique()

2246

In [236]:
#The minimum and maximum values.
print(f"min seller_zip_code_prefix: {min(sellers['seller_zip_code_prefix'])}")
print(f"max seller_zip_code_prefix: {max(sellers['seller_zip_code_prefix'])}")

min seller_zip_code_prefix: 1001
max seller_zip_code_prefix: 99730


In [237]:
#Values below 0 or above 99,999.
print(f"values_below_0: {len(sellers[sellers['seller_zip_code_prefix'] < 0])}")
print(f"values_above_99,999: {len(sellers[sellers['seller_zip_code_prefix'] > 99999])}")

values_below_0: 0
values_above_99,999: 0


In [238]:
#The number of records with a prefix below 10,000
print(f"number of records with prefix below 10,000: {len(sellers[sellers['seller_zip_code_prefix'] < 10000])}")

number of records with prefix below 10,000: 1027


In [239]:
#ZIP-code prefixes associated with more than one seller state.
zip_code_with_state = sellers.groupby('seller_zip_code_prefix')['seller_state'].nunique()
int((zip_code_with_state>1).sum())

17

In [240]:
#Retain only the ZIP prefixes whose distinct-state count is greater than 1.
zip_greater_2 = zip_code_with_state.loc[zip_code_with_state>1]
zip_greater_2

seller_zip_code_prefix
21210    2
22783    2
31160    2
36010    2
37540    2
37795    2
44600    2
80240    2
81020    2
81560    2
83020    2
83321    2
85960    3
88075    2
88301    2
89052    2
95076    2
Name: seller_state, dtype: int64

In [241]:
#Use those prefixes to display the corresponding seller records.
sellers_records_zipcode_with_multiple_state = sellers[sellers['seller_zip_code_prefix'].isin(zip_greater_2.index)].sort_values(by='seller_zip_code_prefix')
sellers_records_zipcode_with_multiple_state.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
114,392f7f2c797e4dc077e4311bde2ab8ce,21210,rio de janeiro,RN
1591,95e03ca3d4146e4011985981aeb959b9,21210,rio de janeiro,RJ
364,c716e0b86ed568878475b60fbb6323ad,22783,rio de janeiro,SP
299,b026643914f651e8371425e5b8a1e780,22783,rio de janeiro,RJ
1693,2c00c85d30361cd2ced2969cffbbffa3,31160,belo horizonte,MG


In [242]:
#Count all rows in sellers_records_zipcode_with_multiple_state.
len(sellers_records_zipcode_with_multiple_state)

45

In [243]:
#Display the complete filtered table, focusing on ZIP prefix, city, and state.
sellers_records_zipcode_with_multiple_state.sort_values(by=['seller_zip_code_prefix', 'seller_city', 'seller_state'])

,seller_id,seller_zip_code_prefix,seller_city,seller_state
1591,95e03ca3d4146e4011985981aeb959b9,21210,rio de janeiro,RJ
114,392f7f2c797e4dc077e4311bde2ab8ce,21210,rio de janeiro,RN
299,b026643914f651e8371425e5b8a1e780,22783,rio de janeiro,RJ
364,c716e0b86ed568878475b60fbb6323ad,22783,rio de janeiro,SP
1693,2c00c85d30361cd2ced2969cffbbffa3,31160,belo horizonte,MG
3036,6b9b80d53ba3676eafe60268a810b5a1,31160,belo horizonte,SP
2509,29e6e05b65f93bc224c644319dc52ecf,36010,juiz de fora,MG
3012,f789d2c4f2c2eb38fc4373e7a4b35264,36010,juiz de fora,MG
680,d9d43faf741ceaa8da52fdbb81d9e0e3,36010,juiz de fora,SP
1299,57df9869a600bd6b7c405f2a862eccfb,37540,santa rita do sapucai,MG


#### Seller Geographic Findings and Treatment Decision

The Sellers table contains **23 distinct state codes**. All state values consist of exactly two uppercase characters, and no blank values were found. São Paulo (`SP`) contains the largest number of sellers, with 1,849 records.

The table contains **611 distinct seller-city values**, with no blank entries or leading or trailing spaces.

There are **2,246 distinct seller ZIP-code prefixes**, ranging from 1,001 to 99,730. No prefixes fall below 0 or above 99,999. However, **1,027 seller records** have prefixes below 10,000. These values are not invalid, but their leading zeros were lost because the field was loaded as an integer. During cleaning, ZIP-code prefixes will be converted to five-character strings and padded with leading zeros.

A total of **17 ZIP-code prefixes are associated with more than one seller state**, affecting **45 seller records**. Inspection revealed potentially inconsistent city–state combinations, including `rio de janeiro` associated with `RN`, `caxias do sul` associated with `SP`, and several Paraná and Santa Catarina cities associated with `SP`.

These records will not be corrected solely from the Sellers table. They will be flagged and compared with the Geolocation table to determine a consistent ZIP-to-state mapping before geographic seller analysis.


## 8. Geolocation Table Audit

The Geolocation table will be audited to evaluate ZIP-code coverage, coordinate validity, duplicate records, and city–state consistency. Because a ZIP-code prefix may contain multiple coordinate observations, the table’s natural grain and duplication patterns must be understood before creating a reusable geographic reference.

### 8.1 Dataset Shape and Structure

The number of rows and columns, column names, data types, and non-null counts will first be inspected to understand the structure of the Geolocation dataset.


In [244]:
geolocation = pd.read_csv(Raw_Data_Path/'olist_geolocation_dataset.csv')
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [245]:
geolocation.shape

(1000163, 5)

In [246]:
geolocation.columns

Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str')

In [247]:
geolocation.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

In [248]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


#### Initial Structure Observations

The Geolocation table contains **1,000,163 rows and 5 columns**. Each row represents a geographic observation containing a ZIP-code prefix, latitude, longitude, city, and state.

The table is not expected to contain one row per ZIP-code prefix because the same prefix may have multiple coordinate observations. Therefore, repeated ZIP prefixes should not automatically be treated as duplicate records.

The ZIP-code prefix is stored as an integer, so prefixes beginning with zero may have lost their leading zeros. Latitude and longitude are stored as numeric values, while city and state are stored as strings.

The structure output shows complete non-null counts, but a formal missing-value summary will be created to confirm this.

### 8.2 Missing-Value Validation

Missing values will be counted and expressed as a percentage of all Geolocation records to confirm whether each geographic field is complete.


In [249]:
geolocation_missing_count = geolocation.isna().sum()
geolocation_missing_precentage = (geolocation_missing_count/len(geolocation))*100
geolocation_missing_summary = pd.DataFrame({"missing_count":geolocation_missing_count, "missing_percentage":geolocation_missing_precentage})
geolocation_missing_summary

,missing_count,missing_percentage
geolocation_zip_code_prefix,0,0.0
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_city,0,0.0
geolocation_state,0,0.0


#### Missing-Value Findings

The Geolocation table contains no missing values. All 1,000,163 records have complete ZIP-code prefixes, latitude, longitude, city, and state information. No missing-value treatment is required.

### 8.3 Grain and Duplicate Validation

The Geolocation table may contain multiple legitimate observations for the same ZIP-code prefix. The audit will distinguish repeated ZIP prefixes from exact duplicate rows and determine the number of unique geographic locations represented.


In [250]:
#The number of distinct ZIP-code prefixes
geolocation['geolocation_zip_code_prefix'].nunique()

19015

In [251]:
#The number of exact duplicate rows.
int(geolocation.duplicated().sum())

261831

In [252]:
#The number of unique complete rows after removing exact duplicates.
geolocation_new = geolocation.drop_duplicates()
len(geolocation_new)

738332

In [253]:
#Calculate exact duplicates as a percentage of all Geolocation rows
geolocation_duplicate_percentage = ((len(geolocation)-len(geolocation_new))/len(geolocation))*100
geolocation_duplicate_percentage

26.17883285024541

In [254]:
#Count the number of unique geographic observations associated with each ZIP-code prefix.
geolocation_observations_per_zip  = geolocation_new.groupby('geolocation_zip_code_prefix').size()
geolocation_observations_per_zip 

geolocation_zip_code_prefix
1001     11
1002      6
1003     11
1004     14
1005     13
         ..
99960     5
99965     6
99970    16
99980    21
99990     2
Length: 19015, dtype: int64

In [255]:
#ZIP prefixes have an observation count greater than 1
geo_zip_greater_1 = int((geolocation_observations_per_zip>1).sum())
geo_zip_greater_1

17823

In [256]:
#maximum observation count in geolocation_observations_per_zip 
max(geolocation_observations_per_zip)

779

#### Grain and Duplicate Findings

The Geolocation table contains **19,015 distinct ZIP-code prefixes** across 1,000,163 rows. A total of **261,831 rows (26.18%)** are exact duplicates, leaving **738,332 unique complete geographic observations**.

After exact duplicates are removed, **17,823 ZIP-code prefixes** still contain more than one unique geographic observation. One ZIP prefix contains as many as **779 unique observations**. These repeated observations are expected because a ZIP-code prefix can cover multiple coordinate points.

Therefore, exact duplicate rows can be removed without information loss, but records must not be deduplicated using the ZIP-code prefix alone. A representative geographic reference will later be created by aggregating coordinates and selecting consistent city–state mappings for each prefix.

### 8.4 Coordinate Range Validation

Latitude and longitude values will be examined for invalid global ranges and unusual observations outside the expected geographic boundaries of Brazil before representative coordinates are calculated.


In [257]:
#Generate descriptive statistics for latitude and longitude using the deduplicated Geolocation table.

geolocation_new[['geolocation_lat', 'geolocation_lng']].describe()


,geolocation_lat,geolocation_lng
count,738332.000000,738332.000000
mean,-20.998353,-46.461098
std,5.892315,4.393705
min,-36.605374,-101.466766
25%,-23.603061,-48.867822
50%,-22.873588,-46.647278
75%,-19.923336,-43.836974
max,45.065933,121.105394


In [258]:
#Count latitude values outside −90 to 90.
print(f"number of latitude values outside -90: {len(geolocation_new[geolocation_new['geolocation_lat'] < -90])}")
print(f"number of latitude values outside 90: {len(geolocation_new[geolocation_new['geolocation_lat'] > 90])}")

number of latitude values outside -90: 0
number of latitude values outside 90: 0


In [259]:
#Count longitude values outside −180 to 180.
print(f"number of longitude values outside -180: {len(geolocation_new[geolocation_new['geolocation_lng'] < -180])}")
print(f"number of longitude values outside 180: {len(geolocation_new[geolocation_new['geolocation_lng'] > 180])}")


number of longitude values outside -180: 0
number of longitude values outside 180: 0


In [260]:
#Filter records where latitude is below −34 or above 6, or longitude is below −74 or above −34

geoloc_filtered = geolocation_new.loc[(geolocation_new['geolocation_lat'] < -34) | (geolocation_new['geolocation_lat'] > 6) | (geolocation_new['geolocation_lng'] < -74) | (geolocation_new['geolocation_lng'] > -32)]
geoloc_filtered

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [261]:
#Count these records.
len(geoloc_filtered)

27

In [262]:
#The number of distinct ZIP-code prefixes among those 27 records.
geoloc_filtered['geolocation_zip_code_prefix'].nunique()

20

In [263]:
#Their percentage of the 738,332 deduplicated Geolocation rows.
geo_fiter_percentage = (len(geoloc_filtered)/len(geolocation_new))*100
geo_fiter_percentage

0.0036568914797137334

In [264]:
unique_geo_filter = geoloc_filtered['geolocation_zip_code_prefix'].unique()
unique_geo_filter

array([18243, 28165, 28155, 28333, 28595, 29654, 35179, 45936, 46560,
       47310, 57319, 58441, 68275, 68379, 68447, 78131, 83252, 83810,
       95130, 98780])

In [265]:
geoloc_affected20_zip_prefix = geolocation_new[geolocation_new['geolocation_zip_code_prefix'].isin(unique_geo_filter)]
geoloc_affected20_zip_prefix

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513637,28165,-24.576786,-53.795538,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513667,28165,-23.282553,-47.292447,vila nova de campos,RJ
...,...,...,...,...,...
993494,98780,-27.863935,-54.480055,santa rosa,RS
993499,98780,-29.937067,-51.025864,santa rosa,RS
993515,98780,-27.870662,-54.486393,santa rosa,RS
993517,98780,-27.863292,-54.471544,santa rosa,RS


In [266]:
valid_coordinates_for_affected_zips = geoloc_affected20_zip_prefix[(geoloc_affected20_zip_prefix['geolocation_lat'] >= -34) & (geoloc_affected20_zip_prefix['geolocation_lat'] <= 6) & (geoloc_affected20_zip_prefix['geolocation_lng'] >= -74) & (geoloc_affected20_zip_prefix['geolocation_lng'] <= -32)]
valid_coordinates_for_affected_zips.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
513637,28165,-24.576786,-53.795538,vila nova de campos,RJ
513667,28165,-23.282553,-47.292447,vila nova de campos,RJ
513678,28155,-19.936270,-44.012515,santa maria,RJ
513684,28165,-30.138855,-51.212135,vila nova de campos,RJ
513702,28155,-22.734149,-43.379124,santa maria,RJ


In [267]:
len(valid_coordinates_for_affected_zips)

256

In [268]:
unique_valid_coordinates_for_affected_zips = valid_coordinates_for_affected_zips['geolocation_zip_code_prefix'].unique()
unique_valid_coordinates_for_affected_zips

array([28165, 28155, 28333, 28595, 29654, 35179, 45936, 46560, 47310,
       57319, 58441, 68275, 68379, 68447, 83810, 98780])

In [269]:
#distinct geolocation_zip_code_prefix values within valid_coordinates_for_affected_zips
valid_coordinates_for_affected_zips['geolocation_zip_code_prefix'].nunique()

16

In [270]:
 pd.Series(unique_geo_filter)

0     18243
1     28165
2     28155
3     28333
4     28595
5     29654
6     35179
7     45936
8     46560
9     47310
10    57319
11    58441
12    68275
13    68379
14    68447
15    78131
16    83252
17    83810
18    95130
19    98780
dtype: int64

In [271]:
missing4_zips = pd.Series(unique_geo_filter)

missing4_zips = missing4_zips.loc[~missing4_zips.isin(unique_valid_coordinates_for_affected_zips)]
missing4_zips

0     18243
15    78131
16    83252
18    95130
dtype: int64

In [272]:
geoloc_filtered[geoloc_filtered['geolocation_zip_code_prefix'].isin(missing4_zips)]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
822529,78131,38.816816,-9.394625,varzea grande,MT
860562,83252,42.184003,-8.723762,ilha dos valadares,PR
965687,95130,14.585073,121.105394,santa lucia do piai,RS


#### Coordinate Findings and Treatment Decision

All latitude and longitude values fall within globally valid coordinate ranges. However, the original data contains observations outside Brazil’s expected geographic boundaries.

After accounting for the legitimate island coordinates of Fernando de Noronha, **27 records across 20 ZIP-code prefixes** remain outside the refined Brazil range. These represent approximately **0.0037%** of the 738,332 deduplicated geographic observations.

Of the 20 affected ZIP prefixes:

* 16 have other valid coordinate observations within Brazil.
* 4 have no valid alternative coordinates: `18243`, `78131`, `83252`, and `95130`.

The four uncovered prefixes are associated with valid-looking Brazilian city and state information, but their coordinates point to locations outside Brazil. The 27 invalid coordinate observations will be excluded from coordinate aggregation. For the 16 prefixes with valid alternatives, representative coordinates can be calculated from their remaining observations. For the four uncovered prefixes, ZIP, city, and state information will be retained, but latitude and longitude will remain unavailable rather than being arbitrarily imputed.

### 8.5 City and State Consistency Validation

City and state values will be checked for formatting issues and ZIP prefixes associated with conflicting geographic labels. These results will also be used to investigate the seller ZIP-to-state inconsistencies identified earlier.


In [273]:
#The number and frequency of distinct states.
distinc_state_frequency = geolocation_new['geolocation_state'].value_counts()
distinc_state_frequency

geolocation_state
SP    285976
MG    101353
RJ     78836
RS     48093
PR     45059
SC     30191
BA     27720
GO     15601
PE     13162
ES     12632
CE      9541
MT      9374
DF      9080
MS      8594
PA      8551
MA      6277
PB      4787
RN      4014
PI      3592
AL      3415
TO      2977
SE      2653
RO      2523
AM      1986
AC      1039
AP       738
RR       568
Name: count, dtype: int64

In [274]:
#Blank or incorrectly formatted state values.
geoloc_state_blank_or_incorrectformat = len(geolocation_new[(geolocation_new['geolocation_state'].str.strip() =='') | (geolocation_new['geolocation_state'].str.len() != 2) | (~geolocation_new['geolocation_state'].str.isupper())])
geoloc_state_blank_or_incorrectformat

0

In [275]:
#Number of distinct city values.
geolocation_new['geolocation_city'].nunique()

8011

In [276]:
#Blank or whitespace-only city values.
geoloc_blank_city = len(geolocation_new[(geolocation_new['geolocation_city'].str.strip() =='')])
geoloc_blank_city

0

In [277]:
#City values containing leading or trailing spaces.
print(f"city value containing leading space: {int(geolocation_new['geolocation_city'].str.startswith(' ').sum())}")
print(f"city value containing trailing space: {int(geolocation_new['geolocation_city'].str.endswith(' ').sum())}")

city value containing leading space: 0
city value containing trailing space: 1


In [278]:
city_with_trailing_space = geolocation_new[geolocation_new['geolocation_city'].str.endswith(' ')]
city_with_trailing_space

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
670078,40243,-12.987814,-38.505742,salvador,BA


In [279]:
#Geolocation ZIP prefixes map to multiple states
geoloc_zip_multi_state = geolocation_new.groupby('geolocation_zip_code_prefix')['geolocation_state'].nunique()
geoloc_zip_multi_state

geolocation_zip_code_prefix
1001     1
1002     1
1003     1
1004     1
1005     1
        ..
99960    1
99965    1
99970    1
99980    1
99990    1
Name: geolocation_state, Length: 19015, dtype: int64

In [280]:
#How many values are greater than 1.
int((geoloc_zip_multi_state>1).sum())

8

In [281]:
#maximum number of states associated with one zip_prefix
max(geoloc_zip_multi_state)

2

In [282]:
geoloc_zip_greater1 = geoloc_zip_multi_state[geoloc_zip_multi_state>1]
geoloc_zip_greater1

geolocation_zip_code_prefix
2116     2
4011     2
21550    2
23056    2
72915    2
78557    2
79750    2
80630    2
Name: geolocation_state, dtype: int64

In [283]:
#inspect those eight prefixes:1.Retain ZIP prefixes whose distinct-state count is greater than 1
geoloc_multi_state = geolocation_new[geolocation_new['geolocation_zip_code_prefix'].isin(geoloc_zip_greater1.index)]
geoloc_multi_state.groupby('geolocation_zip_code_prefix')['geolocation_state'].unique()

geolocation_zip_code_prefix
2116     [SP, RN]
4011     [SP, AC]
21550    [RJ, AC]
23056    [RJ, AC]
72915    [GO, DF]
78557    [MT, RO]
79750    [MS, RS]
80630    [PR, SC]
Name: geolocation_state, dtype: object

In [284]:
geoloc_zip_state_frequency = (geoloc_multi_state.groupby(["geolocation_zip_code_prefix","geolocation_state"]).size().reset_index(name="record_count")
                              .sort_values(by=["geolocation_zip_code_prefix","record_count"],ascending=[True, False]))

geoloc_zip_state_frequency

,geolocation_zip_code_prefix,geolocation_state,record_count
1,2116,SP,10
0,2116,RN,1
3,4011,SP,69
2,4011,AC,1
5,21550,RJ,144
4,21550,AC,1
7,23056,RJ,30
6,23056,AC,1
9,72915,GO,9
8,72915,DF,1


In [285]:
#17 Seller ZIP conflicts

seller_zip_in_geoloc = geolocation_new[geolocation_new['geolocation_zip_code_prefix'].isin(zip_greater_2.index)]

seller_zip_in_geoloc.groupby(["geolocation_zip_code_prefix","geolocation_state"]).size()

geolocation_zip_code_prefix  geolocation_state
21210                        RJ                   167
22783                        RJ                    98
31160                        MG                   166
36010                        MG                   211
37540                        MG                   289
37795                        MG                   156
44600                        BA                    61
80240                        PR                   307
81020                        PR                    98
81560                        PR                    66
83020                        PR                    78
83321                        PR                    66
85960                        PR                   267
88075                        SC                   103
88301                        SC                   164
89052                        SC                    65
95076                        RS                    58
dtype: int64

In [286]:
seller_zip_reference_state = (seller_zip_in_geoloc.groupby("geolocation_zip_code_prefix")["geolocation_state"].first())
seller_zip_reference_state

geolocation_zip_code_prefix
21210    RJ
22783    RJ
31160    MG
36010    MG
37540    MG
37795    MG
44600    BA
80240    PR
81020    PR
81560    PR
83020    PR
83321    PR
85960    PR
88075    SC
88301    SC
89052    SC
95076    RS
Name: geolocation_state, dtype: str

In [287]:
seller_state_comparison = (sellers_records_zipcode_with_multiple_state.copy())

seller_state_comparison["reference_state"] = (seller_state_comparison["seller_zip_code_prefix"].map(seller_zip_reference_state))

seller_state_comparison

,seller_id,seller_zip_code_prefix,seller_city,seller_state,reference_state
114,392f7f2c797e4dc077e4311bde2ab8ce,21210,rio de janeiro,RN,RJ
1591,95e03ca3d4146e4011985981aeb959b9,21210,rio de janeiro,RJ,RJ
364,c716e0b86ed568878475b60fbb6323ad,22783,rio de janeiro,SP,RJ
299,b026643914f651e8371425e5b8a1e780,22783,rio de janeiro,RJ,RJ
1693,2c00c85d30361cd2ced2969cffbbffa3,31160,belo horizonte,MG,MG
3036,6b9b80d53ba3676eafe60268a810b5a1,31160,belo horizonte,SP,MG
2509,29e6e05b65f93bc224c644319dc52ecf,36010,juiz de fora,MG,MG
680,d9d43faf741ceaa8da52fdbb81d9e0e3,36010,juiz de fora,SP,MG
3012,f789d2c4f2c2eb38fc4373e7a4b35264,36010,juiz de fora,MG,MG
1299,57df9869a600bd6b7c405f2a862eccfb,37540,santa rita do sapucai,MG,MG


In [288]:
seller_state_mismatches = seller_state_comparison.loc[seller_state_comparison["seller_state"]!= seller_state_comparison["reference_state"]]

seller_state_mismatches

,seller_id,seller_zip_code_prefix,seller_city,seller_state,reference_state
114,392f7f2c797e4dc077e4311bde2ab8ce,21210,rio de janeiro,RN,RJ
364,c716e0b86ed568878475b60fbb6323ad,22783,rio de janeiro,SP,RJ
3036,6b9b80d53ba3676eafe60268a810b5a1,31160,belo horizonte,SP,MG
680,d9d43faf741ceaa8da52fdbb81d9e0e3,36010,juiz de fora,SP,MG
890,2a167ca73899c85001a837d8fb4962f6,37540,sao paulo,SP,MG
1656,f00e21b1e91a79653163b7fd8f293ff1,37795,andradas,SP,MG
2048,b43d8b707f887a30f4675391098354a0,44600,ipira,SP,BA
1905,296729ffb9b684050dd24836dac4494a,80240,curitiba,SP,PR
2510,cce6ab8d1682639fe45ab70234f1665f,81020,curitiba,SP,PR
1233,a3082f442524a1be452e3189e003b361,81560,curitiba,SP,PR


In [289]:
len(seller_state_mismatches)

19

In [290]:
#Minimum and Maximum ZIP-code prefix
print(f"min zip_code_prefix in geolocation: {min(geolocation_new['geolocation_zip_code_prefix'])}")

print(f"max zip_code_prefix in geolocation: {max(geolocation_new['geolocation_zip_code_prefix'])}")

min zip_code_prefix in geolocation: 1001
max zip_code_prefix in geolocation: 99990


In [291]:
#Number of records below 0
len(geolocation_new[geolocation_new['geolocation_zip_code_prefix'] < 0])

0

In [292]:
#Number of records above 99,999
len(geolocation_new[geolocation_new['geolocation_zip_code_prefix'] > 99999])

0

In [293]:
#Number of records with ZIP prefixes below 10,000
len(geolocation_new[geolocation_new['geolocation_zip_code_prefix'] < 10000])

158832

#### City, State, and ZIP-Code Findings

The deduplicated Geolocation table contains all **27 Brazilian state codes**. No state values are missing, blank, incorrectly sized, or improperly formatted.

There are **8,011 distinct city labels**. No blank or leading-space values were found, but one `salvador` record contains a trailing space. City values will be trimmed during cleaning to avoid unnecessary label duplication.

Eight ZIP-code prefixes are associated with two states. In every case, one state has many observations while the conflicting state appears only once. Therefore, the most frequent state will be used as the canonical ZIP-to-state mapping.

The 17 conflicting Seller ZIP prefixes were compared with Geolocation. Geolocation provides one unambiguous state for each prefix, and **19 of the 45 affected Seller records** contain a state that differs from the reference state. These 19 Seller state values will be corrected in the cleaned dataset using the Geolocation ZIP-to-state mapping, while the raw values will remain unchanged.

Geolocation ZIP-code prefixes range from **1,001 to 99,990**, with no values below 0 or above 99,999. However, **158,832 deduplicated records** have prefixes below 10,000, indicating that leading zeros were lost when the field was loaded as an integer. ZIP prefixes will be converted to five-character strings and padded with leading zeros during cleaning.


## 9. Cross-Table Referential Integrity Validation

Foreign-key relationships between the Olist tables will be validated to ensure that transactional records have matching parent records. Orphaned identifiers and missing child-table coverage will be counted before the analytical dataset is created.

### 9.1 Orders–Customers Relationship

The `customer_id` relationship between Orders and Customers will be checked in both directions to confirm whether every order has a matching customer record and whether every customer record is associated with an order.


In [294]:
#How many customer_id values in Orders do not exist in Customers.

customerid_in_orders_notin_customers = orders[~orders['customer_id'].isin(customers['customer_id'])]
len(customerid_in_orders_notin_customers)

0

In [295]:
#How many customer_id values in Customers do not exist in Orders.
customerid_in_customers_notin_orders = customers[~customers['customer_id'].isin(orders['customer_id'])]
len(customerid_in_customers_notin_orders)

0

#### Orders–Customers Findings

No orphaned `customer_id` values were found in either direction. Every order has a matching Customer record, and every Customer record is associated with an order.

The relationship is complete at the `customer_id` level. Because each `customer_id` identifies one order-specific customer record, `customer_unique_id` will be used instead when identifying repeat customers across multiple orders.

### 9.2 Orders–Order Items Relationship

The `order_id` relationship between Orders and Order Items will be validated in both directions. Order-item records without a matching order would represent foreign-key violations, while orders without item records may reflect particular order lifecycle statuses and require investigation.


In [296]:
#How many order_id values in Order Items do not exist in Orders.

orderid_in_orderitems_notin_orders = order_items[~order_items['order_id'].isin(orders['order_id'])]
len(orderid_in_orderitems_notin_orders)

0

In [297]:
#How many order_id values in Orders do not exist in Order Items.
orderid_in_orders_notin_orderitems = orders[~orders['order_id'].isin(order_items['order_id'])]
len(orderid_in_orders_notin_orderitems)

775

In [298]:
#order_status frequency within orderid_in_orders_notin_orderitems

orderid_in_orders_notin_orderitems['order_status'].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [299]:
#inspect that shipped order from orderid_in_orders_notin_orderitems.
shipped_notin_order_items =orderid_in_orders_notin_orderitems[orderid_in_orders_notin_orderitems['order_status']=="shipped"]
shipped_notin_order_items

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
23254,a68ce1686d536ca72bd2dadc4b8671e5,d7bed5fac093a4136216072abaf599d5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22,2016-11-07 16:37:37,NaN,2016-12-01 00:00:00


#### Orders–Order Items Findings

No Order Items records contain an `order_id` absent from the Orders table, confirming that the Order Items foreign key is valid.

However, **775 orders** have no matching item records. Their status distribution is:

* Unavailable: 603
* Canceled: 164
* Created: 5
* Invoiced: 2
* Shipped: 1

The unavailable, canceled, created, and invoiced orders may legitimately lack completed item records because they did not progress through the full order lifecycle.

One shipped order has no Order Items record despite having approval and carrier-delivery timestamps. This record is classified as a cross-table data-quality exception. It can remain in order-status reporting but cannot be used in item-level, product-level, or sales-value analysis. It is outside the delivered-order analytical scope and will be flagged rather than removed from the raw data.

### 9.3 Order Items–Products and Sellers Relationships

The `product_id` and `seller_id` foreign keys in Order Items will be checked against the Products and Sellers parent tables. Parent-table records without transactions will also be counted to understand catalogue and seller coverage.


In [300]:
#Order Items product IDs absent from Products.
productid_in_orderitems_notin_products = order_items[~order_items['product_id'].isin(products['product_id'])]
len(productid_in_orderitems_notin_products)

0

In [301]:
#Product IDs absent from Order Items.
productid_in_products_notin_orderitems = products[~products['product_id'].isin(order_items['product_id'])]
len(productid_in_products_notin_orderitems)

0

In [302]:
#Order Items seller IDs absent from Sellers.
sellerid_in_orderitems_notin_sellers = order_items[~order_items['seller_id'].isin(sellers['seller_id'])]
len(sellerid_in_orderitems_notin_sellers)

0

In [303]:
#Seller IDs absent from Order Items.
sellerid_in_sellers_notin_orderitems = sellers[~sellers['seller_id'].isin(order_items['seller_id'])]
len(sellerid_in_sellers_notin_orderitems)

0

#### Order Items–Products and Sellers Findings

No orphaned `product_id` or `seller_id` values were found in Order Items. Every order-item product has a matching Products record, and every order-item seller has a matching Sellers record.

All Product records appear in Order Items, and all Seller records are associated with at least one Order Items transaction. Therefore, the Product and Seller parent tables have complete transactional coverage, and both foreign-key relationships are valid.

### 9.4 Orders–Payments Relationship

The `order_id` relationship between Orders and Payments will be checked in both directions. Payment records without matching orders would represent foreign-key violations, while orders without payment records may require lifecycle investigation.


In [304]:
#How many Payment order_id values do not exist in Orders.
orderid_in_payments_notin_orders = payments[~payments['order_id'].isin(orders['order_id'])]
len(orderid_in_payments_notin_orders)

0

In [305]:
#How many Order order_id values do not exist in Payments
orderid_in_orders_notin_payments = orders[~orders['order_id'].isin(payments['order_id'])] 
len(orderid_in_orders_notin_payments)

1

In [306]:
orderid_in_orders_notin_payments

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00


In [307]:
#inspect the corresponding Order Items records and determine:
#Number of item records.
#Total merchandise price.
#Total freight value.

missing_payment_record_orderid_orderitems = order_items[order_items['order_id'].isin(orderid_in_orders_notin_payments['order_id'])]
print(f"Number of item recordsin order_items: {len(missing_payment_record_orderid_orderitems)}")
print(f"Total merchandise price: {float(missing_payment_record_orderid_orderitems['price'].sum())}")
print(f"Total merchandise price: {float(missing_payment_record_orderid_orderitems['freight_value'].sum())}")


Number of item recordsin order_items: 3
Total merchandise price: 134.97
Total merchandise price: 8.49


#### Orders–Payments Findings

No Payment records contain an `order_id` absent from the Orders table, confirming that the Payments foreign key is valid.

One delivered order has no corresponding Payment record. The order contains valid approval and delivery timestamps and includes **3 item records**, with a total merchandise price of **134.97** and freight value of **8.49**.

This order is classified as a cross-table data-quality exception because a completed order would normally have payment information. It can remain in delivered-order and merchandise-sales analysis because its item prices are available, but it will be flagged and excluded from payment-method, payment-value, and payment-reconciliation analysis.

### 9.5 Orders–Reviews Relationship

The `order_id` relationship between Orders and Reviews will be checked in both directions. Review records without matching orders would represent foreign-key violations, while orders without reviews may be expected because customers are not required to submit feedback.


In [308]:
#How many Review order_id values do not exist in Orders.
orderid_in_reviews_notin_orders = reviews[~reviews['order_id'].isin(orders['order_id'])]
len(orderid_in_reviews_notin_orders)

0

In [309]:
#How many Order order_id values do not exist in Reviews.
orderid_in_orders_notin_reviews = orders[~orders['order_id'].isin(reviews['order_id'])]
len(orderid_in_orders_notin_reviews)

768

In [310]:
orderid_in_orders_notin_reviews['order_status'].value_counts()

order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64

#### Orders–Reviews Findings

No Review records contain an `order_id` absent from the Orders table, confirming that the Reviews foreign key is valid.

A total of **768 orders** have no Review record. Their status distribution is:

* Delivered: 646
* Shipped: 75
* Canceled: 20
* Unavailable: 14
* Processing: 6
* Invoiced: 5
* Created: 2

Missing reviews are not data-quality errors because customers are not required to submit feedback. The 646 delivered orders without reviews will remain in sales and delivery analysis. They will be represented separately as `no_review` when comparing customer satisfaction, rather than being removed.

### 9.6 Join Cardinality and Row-Count Risks

The cross-table audit identified the following relationship structure:

* Orders to Customers is one-to-one through `customer_id`.
* Orders to Order Items is one-to-many through `order_id`.
* Order Items to Products is many-to-one through `product_id`.
* Order Items to Sellers is many-to-one through `seller_id`.
* Orders to Payments is one-to-many because an order may contain multiple payment records.
* Orders to Reviews is optional and may be one-to-many because some orders have no review while 547 orders have multiple review IDs.

Customers, Products, and Sellers can be joined to their child tables without increasing row counts beyond the expected transactional grain because their join keys are unique.

Raw Payments and Reviews must not be joined directly to the order-item dataset because multiple records per order could multiply rows and overstate sales. Payments will first be aggregated to the order level. Reviews will first be reduced or aggregated to one documented order-level representation.

The analytical dataset will begin at the delivered order-item grain. Left joins will be used to preserve eligible transactions, and the row count will be checked after every join. Joins to unique parent tables should not increase the base row count, while any unexpected increase will be treated as a join-cardinality error.


## 10. Consolidated Data-Quality Findings and Cleaning Plan

The data-quality audit validated the structure, completeness, identifiers, numeric values, timestamps, geographic fields, and cross-table relationships of the Olist datasets. Raw records will remain unchanged; cleaning decisions will be implemented in a separate preparation notebook.

### Orders

* `order_id` is complete and unique.
* The five timestamp columns will be converted to datetime.
* Missing timestamps are generally consistent with unfinished or canceled orders.
* Delivered-order exceptions include 8 missing customer-delivery dates, 2 missing carrier-delivery dates, and 14 missing approval dates.
* Lifecycle inconsistencies include carrier delivery before purchase or approval and customer delivery before carrier delivery.
* These orders will remain eligible for completed-sales analysis but will be flagged and excluded from calculations requiring the affected timestamps.

### Customers

* `customer_id` is complete and unique.
* `customer_unique_id` is intentionally repeated and will be used to identify repeat customers.
* State and city values are complete and correctly formatted.
* ZIP-code prefixes will be converted to five-character strings and padded with leading zeros.

### Order Items

* The combination of `order_id` and `order_item_id` is complete and unique.
* Item numbering is positive and sequential within each order.
* Price and freight values contain no negative values.
* Zero freight values will be retained as valid free-shipping records.
* `shipping_limit_date` will be converted to datetime.
* Four future shipping-limit records from one seller will be flagged and excluded from shipping-deadline analysis.

### Payments

* The combination of `order_id` and `payment_sequential` is unique.
* Eighty orders have incomplete payment sequences and will be flagged rather than renumbered.
* Three `not_defined` zero-value payments belong to canceled orders and will be excluded from payment-method analysis.
* Two delivered orders contain zero installments and will be excluded from installment analysis.
* Zero-value voucher rows will be retained with their associated positive payment records.
* One delivered order has no Payment record despite containing 3 items with merchandise and freight value totaling 143.46. It will remain in merchandise-sales analysis but be excluded from payment analysis.
* Payments will be aggregated to the order level before being joined to an order-item dataset.

### Reviews

* Required identifiers, scores, and timestamps are complete.
* Missing review titles and messages are expected because written comments are optional.
* Review scores are valid values from 1 through 5.
* Timestamp conversion introduced no parsing failures, and no answer timestamp occurs before its creation date.
* A total of 682 reviews have response durations greater than 30 days and will be flagged for response-time analysis.
* Neither `review_id` nor `order_id` is independently unique, although their combination is unique.
* There are 547 orders with multiple review IDs and 768 orders without reviews, including 646 delivered orders.
* Reviews will be reduced or aggregated to one documented order-level representation before joining. Orders without reviews will remain and be labeled `no_review`.

### Products and Category Translation

* `product_id` is complete and unique.
* Column-name spelling will be corrected from `lenght` to `length`.
* A total of 610 products are missing category and content attributes. They affect 1,603 item records and will be retained with an `unknown` category.
* Two products are missing weight and dimensions and will be excluded only from measurement-based calculations.
* Four products have zero weight. Their weights will be treated as missing without removing their transactions.
* High weight and dimension values were inspected and will be retained as plausible outliers.
* Two categories lack supplied English translations. Documented manual translations will be added for `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`.

### Sellers

* `seller_id` is complete and unique.
* Seller city and state fields contain no missing or formatting errors.
* ZIP-code prefixes will be converted to five-character strings and padded with leading zeros.
* Nineteen Seller records contain states inconsistent with their Geolocation ZIP mapping. These states will be corrected in the cleaned dataset while preserving the raw values for traceability.

### Geolocation

* The table contains no missing values.
* A total of 261,831 exact duplicate rows will be removed from the cleaned geographic reference.
* Multiple unique coordinate observations per ZIP prefix will be retained until representative coordinates are calculated.
* Twenty-seven coordinate observations fall outside Brazil’s expected range. Sixteen affected ZIP prefixes have valid alternatives; four prefixes will retain city and state information without representative coordinates.
* One city value contains trailing whitespace, which will be trimmed.
* Eight ZIP prefixes contain conflicting state labels. The dominant state will be used as the canonical mapping.
* ZIP-code prefixes will be converted to five-character strings and padded with leading zeros.

### Cross-Table and Join Rules

* No orphaned foreign keys were found between Orders, Customers, Order Items, Products, Sellers, Payments, and Reviews.
* A total of 775 orders have no item records. Most are unavailable, canceled, created, or invoiced; one shipped order without items will be flagged as an exception.
* The analytical base will use delivered order items.
* Left joins will preserve eligible transactions and orders without optional reviews.
* Customers, Products, and Sellers can be joined using their unique identifiers without increasing the expected row count.
* Raw Payments and Reviews will not be joined directly because their one-to-many relationships could multiply rows and overstate sales.
* Row counts and key uniqueness will be validated after every cleaning and joining step.

These decisions form the cleaning specification for the next stage of the project.
